In [1]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, date


#  Smart APS V4  —  Forward-Looking Demand Horizon
#  Planning date  : 14th March 2026
#  Actual col     : "2026-03-12 Total Production Plan"  (supply on 15th)
#  Tentative col  : "2026-03-14 Total Production Plan"  (pre-build for 17th)
#  16th March     : ignored today, planned on 15th when data arrives
# =============================================================

# =============================================================
# SECTION 1 — COLUMN NAMES  (change these every day)
# -------------------------------------------------------------
# On 14th March your demand file has:
#   "2026-03-12 Total Production Plan" → actual for 15th  (D+1)
#   "2026-03-14 Total Production Plan" → tentative for 17th (D+3)
#
# Every morning update the two column name strings below
# to match whatever columns appear in that day's file.
# The planning date is only used for naming the output file.
# =============================================================

ACTUAL_DEMAND_COL    = "2026-03-12 Total Production Plan"   # D+1  ← change daily
TENTATIVE_DEMAND_COL = "2026-03-14 Total Production Plan"   # D+3  ← change daily
PLANNING_DATE        = date(2026, 3, 14)                    # today ← change daily

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS    = 22       # total machine hours per shift
MIN_RUN_HOURS      = 4        # minimum block per part on a machine
TARGET_DAYS_INV    = 3        # ideal inventory buffer (days)
MACHINE_STATE_FILE = "machine_state.json"

# Z-score for tentative demand buffer
# Since your tentative is nearly accurate → Z = 1.28 (90% coverage, modest buffer)
# If tentative were very volatile we'd use 1.65 or 2.0
Z_FOR_TENTATIVE    = 1.28

# Priority weights
W_COVERAGE   = 2.0
W_URGENCY    = 3.0
W_RISK       = 1.5
W_D3_BOOST   = 2.0   # how strongly D+3 pre-build need lifts priority

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path   = "C:/Users/Ex0164/Book1.xlsx"
daily_path  = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
output_path = f"Smart_APS_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — LOAD DATA
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V4  —  Planning date: {PLANNING_DATE}")
print(f"  Actual demand col   : {ACTUAL_DEMAND_COL}")
print(f"  Tentative demand col: {TENTATIVE_DEMAND_COL}")
print(f"  16th March          : IGNORED today — will plan on 15th")
print(f"{'='*62}\n")

print("Loading data...")
stats     = pd.read_excel(book_path,   sheet_name="Sheet2")
daily     = pd.read_excel(daily_path,  sheet_name="Sheet1")
hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

# =============================================================
# SECTION 5 — VALIDATE DEMAND COLUMNS
# -------------------------------------------------------------
# Columns are read directly by name — no date parsing needed.
# If a column is missing, a clear error is raised showing what
# columns are actually in the file so you know what to fix.
# =============================================================

def validate_column(df, col_name, label):
    """Check the column exists; if not, print all columns and raise."""
    if col_name in df.columns:
        print(f"  {label:25s}: '{col_name}'  ✓")
        return
    raise ValueError(
        f"\n  ERROR: Column not found — {col_name}\n"
        f"  Label : {label}\n"
        f"  Fix   : Update {label.split()[0].upper()}_DEMAND_COL in Section 1\n"
        f"  Columns available in file:\n"
        + "\n".join(f"    '{c}'" for c in df.columns)
    )

print("Validating demand columns...")
validate_column(daily, ACTUAL_DEMAND_COL,    "Actual demand col")
validate_column(daily, TENTATIVE_DEMAND_COL, "Tentative demand col")

# =============================================================
# SECTION 6 — MERGE & PRODUCTION RATE
# =============================================================

data = stats.merge(daily, left_on="Part", right_on="Material")

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]
data = data[data["Rate"].notna()].copy()

# =============================================================
# SECTION 7 — BUILD LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        k: (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
    }

inventory       = safe_dict(data, "Material", "Inventory on 24th")
rate            = safe_dict(data, "Material", "Rate")

# D+1: actual demand — must supply this (15th March)
demand_d1       = safe_dict(data, "Material", ACTUAL_DEMAND_COL)

# D+3: tentative demand — pre-build buffer for this (17th March)
demand_d3_tent  = safe_dict(data, "Material", TENTATIVE_DEMAND_COL)

# Historical stats (for smarter Z-score if available)
mean_demand = (safe_dict(data, "Material", "Mean_Demand")
               if "Mean_Demand" in data.columns else demand_d1.copy())
std_demand  = (safe_dict(data, "Material", "Std_Dev_Demand")
               if "Std_Dev_Demand" in data.columns
               else {k: 0.0 for k in demand_d1})

# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# =============================================================

# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# -------------------------------------------------------------
# This file is created automatically after the first run.
# On first run (file does not exist yet):
#   → No changeover penalty applied to anyone — clean slate
#   → After planning, the file is written with today's last parts
# From second run onwards:
#   → File is read, changeover penalties apply correctly
#   → Parts that ran last on a machine get zero changeover cost
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"  Machine state loaded  :")
        for m, p in state.items():
            print(f"    {m:25s} last ran → {p}")
        return state
    # First run — no state file exists yet
    print("  Machine state         : FIRST RUN — no previous state")
    print("                          No changeover penalties applied today.")
    print(f"                          State will be saved to '{MACHINE_STATE_FILE}'")
    print("                          after this run for use tomorrow.")
    return {}   # empty dict → machine_last_part[m] = None → no penalty

def save_machine_state(hz_state, vt_state):
    combined = {**hz_state, **vt_state}
    # Remove machines where last part is None (never ran)
    combined = {m: p for m, p in combined.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → {MACHINE_STATE_FILE}")
    print("  (Tomorrow's plan will use this to avoid unnecessary changeovers)")
    for m, p in combined.items():
        print(f"    {m:25s} last ran → {p}")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

hz_compat, hz_machines = build_compatibility(hz_matrix)
vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — DEMAND HORIZON ANALYSIS
# -------------------------------------------------------------
# For each part, calculate:
#
#   d1_shortage   = max(0, D+1 demand − current inventory)
#                 = what we MUST produce today for 15th supply
#
#   d3_buffer     = extra to pre-build today for 17th demand
#                 = max(0, risk_adjusted_D3 − inv_at_d3_start
#                           − machine_capacity_on_16th
#                           − machine_capacity_on_17th)
#
#   today_target  = d1_shortage + d3_buffer
#
# Why do we subtract capacity on 16th AND 17th?
#   Because even without a demand file for 16th, we can still
#   PRODUCE on 16th. So the pre-build needed today is only what
#   BOTH 16th and 17th combined cannot cover.
#
# Note: Since tentative is nearly accurate, Z = 1.28 (modest buffer)
# =============================================================

def compute_demand_horizon(parts, compatibility, machines):

    def max_daily_capacity(part):
        """Max units producible in one full day across all compatible machines."""
        compat_machines = compatibility.get(part, [])
        if not compat_machines:
            return 0
        r = rate.get(part, 1)
        # Each machine can run AVAILABLE_HOURS independently
        return len(compat_machines) * AVAILABLE_HOURS * r

    rows = []
    for p in parts:
        inv      = inventory.get(p, 0)
        d1       = demand_d1.get(p, 0)
        d3_t     = demand_d3_tent.get(p, 0)
        mean     = mean_demand.get(p, max(d1, 1))
        std      = std_demand.get(p, 0)
        r        = rate.get(p, 1)

        # ── D+1 (15th) shortage ─────────────────────────────
        d1_shortage    = max(0.0, d1 - inv)
        inv_after_d1   = max(0.0, inv - d1)   # inventory left after supplying 15th

        # ── D+3 (17th) risk-adjusted target ─────────────────
        # Since tentative is nearly accurate, small Z = 1.28
        d3_risk_adj    = d3_t + Z_FOR_TENTATIVE * std

        # ── Capacity available on 16th and 17th ─────────────
        # 16th: no demand file, machines run freely → full capacity available
        # 17th: machines run freely → full capacity available
        # Together they can cover: 2 × daily_capacity
        cap_16th       = max_daily_capacity(p)
        cap_17th       = max_daily_capacity(p)

        # Inventory at start of 17th =
        #   inv_after_d1 (left after 15th supply)
        #   + whatever we produce on 16th (up to cap_16th, but only if needed)
        # We assume 16th production goes toward 17th demand as well
        # So total coverage for 17th = inv_after_d1 + cap_16th + cap_17th

        total_coverage_for_d3 = inv_after_d1 + cap_16th + cap_17th

        # Pre-build needed today = gap that even 16th+17th machines can't cover
        d3_buffer = max(0.0, d3_risk_adj - total_coverage_for_d3)

        # ── Combined today's target ──────────────────────────
        today_target = d1_shortage + d3_buffer

        # ── D+3 status flag ─────────────────────────────────
        if d3_buffer > 0:
            d3_flag = "PRE-BUILD NEEDED"
        elif d3_risk_adj > (cap_16th + cap_17th) * 0.8:
            d3_flag = "WATCH — near capacity"
        else:
            d3_flag = "OK"

        rows.append({
            "Part":                   p,
            "Inventory_Now":          round(inv, 0),

            # D+1 (15th) columns
            "D1_Actual_15th":         round(d1, 0),
            "D1_Shortage":            round(d1_shortage, 0),
            "Inv_After_15th_Supply":  round(inv_after_d1, 0),

            # D+3 (17th) columns
            "D3_Tentative_17th":      round(d3_t, 0),
            "D3_Risk_Adjusted":       round(d3_risk_adj, 0),
            "Cap_16th_Available":     round(cap_16th, 0),
            "Cap_17th_Available":     round(cap_17th, 0),
            "D3_Pre_Build_Buffer":    round(d3_buffer, 0),
            "D3_Flag":                d3_flag,

            # Combined
            "Today_Total_Target_Qty": round(today_target, 0),
            "Today_Total_Target_Hrs": round(today_target / r if r > 0 else 0, 2),
        })

    return pd.DataFrame(rows)

# =============================================================
# SECTION 11 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv  = inventory.get(p, 0)
        mean = mean_demand.get(p, demand_d1.get(p, 1))
        coverage.append(inv / mean if mean > 0 else 999)

    n         = len(parts)
    critical  = sum(1 for c in coverage if c < 1)
    low       = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, 2.00, "SCENARIO 0 — ALL parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, 1.65, f"SCENARIO 1 — {critical}/{n} parts critical"
    elif low > 0:
        return 2, 1.28, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, 1.28, f"SCENARIO 3 — All parts healthy (≥{TARGET_DAYS_INV} days)"

# =============================================================
# SECTION 12 — PRIORITY SCORING  (D+3-aware)
# =============================================================

def compute_priority(parts, horizon_df):
    horizon = horizon_df.set_index("Part")
    rows    = []

    for p in parts:
        inv  = inventory.get(p, 0)
        d1   = demand_d1.get(p, 0)
        mean = mean_demand.get(p, max(d1, 1))
        std  = std_demand.get(p, 0)

        days_cov = inv / mean if mean > 0 else 999

        # Urgency based on D+1 coverage
        if days_cov < 1:
            urgency = 1.0
        elif days_cov < 2:
            urgency = 0.5
        else:
            urgency = 0.0

        cv = std / mean if mean > 0 else 0

        # D+3 boost: if pre-build buffer > 0, add extra urgency
        d3_buf  = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
        d3_boost = min(1.0, d3_buf / max(mean, 1))

        score = (W_COVERAGE * (1.0 / (days_cov + 0.01))
               + W_URGENCY  * urgency
               + W_RISK     * cv
               + W_D3_BOOST * d3_boost)

        today_target = (horizon.loc[p, "Today_Total_Target_Qty"]
                        if p in horizon.index else d1)

        rows.append({
            "Part":           p,
            "Days_Coverage":  round(days_cov, 2),
            "Urgency_D1":     urgency,
            "CV":             round(cv, 3),
            "D3_Boost":       round(d3_boost, 3),
            "Score":          round(score, 4),
            "D1_Demand_15th": round(d1, 0),
            "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            "D3_Pre_Build":   round(d3_buf, 0),
            "Today_Target":   round(today_target, 0),
        })

    return (pd.DataFrame(rows)
              .sort_values("Score", ascending=False)
              .reset_index(drop=True))

# =============================================================
# SECTION 13 — UBER-STYLE MACHINE RANKER
# =============================================================

def rank_machines(part, machines, compatibility, machine_hours, machine_last_part):
    """
    Returns machines sorted by assignment cost (lowest = best choice).
    Cost = changeover_penalty + load_factor
    Same part ran last on machine → 0 changeover penalty (zero cost bonus)
    """
    ranked = []
    for m in machines:
        if m not in compatibility.get(part, []):
            continue
        used = machine_hours.get(m, 0)
        free = AVAILABLE_HOURS - used
        if free < MIN_RUN_HOURS:
            continue
        last               = machine_last_part.get(m)
        changeover_penalty = 0.0 if last == part else 0.5
        cost               = changeover_penalty + (used / AVAILABLE_HOURS)
        ranked.append((m, cost, free, changeover_penalty))

    ranked.sort(key=lambda x: x[1])
    return ranked

# =============================================================
# SECTION 14 — 22-HOUR UTILIZATION FILLER
# =============================================================

def fill_remaining_hours(plan, machine_hours, machine_last_part,
                         parts, compatibility, machines,
                         current_inventory, horizon_df, scenario):
    """
    After primary assignments, fill remaining machine time.
    Priority order:
      1. Extend an already-planned part (zero changeover) if it still
         needs more production (D+3 buffer not yet met OR below 3-day inv)
      2. Add a new part — pick the one with the highest D+3 pre-build need
    """
    filler = []
    horizon = horizon_df.set_index("Part")

    for m in machines:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
        if remaining < MIN_RUN_HOURS:
            continue

        on_machine = [row["Part"] for row in plan if row["Machine"] == m]

        # ── Option A: extend already-planned part ────────────
        extended = False
        for p in on_machine:
            inv_now    = current_inventory.get(p, 0)
            target_inv = mean_demand.get(p, demand_d1.get(p, 0)) * TARGET_DAYS_INV
            d3_buf     = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0

            if inv_now < target_inv or d3_buf > 0:
                r_val     = rate.get(p, 1)
                extra_qty = round(remaining * r_val, 0)
                machine_hours[m]       += remaining
                current_inventory[p]    = current_inventory.get(p, 0) + extra_qty

                filler.append({
                    "Part":           p,
                    "Machine":        m,
                    "Run_Hours":      round(remaining, 2),
                    "Production_Qty": extra_qty,
                    "Type":           "Extend — D3 buffer" if d3_buf > 0 else "Extend — inv build",
                    "Changeover":     "No",
                    "D1_Demand_15th": round(demand_d1.get(p, 0), 0),
                    "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
                })
                extended = True
                break

        if extended:
            continue

        # ── Option B: new part — highest D+3 need first ──────
        candidates = []
        for p in parts:
            if p in on_machine:
                continue
            if m not in compatibility.get(p, []):
                continue
            inv_now  = current_inventory.get(p, 0)
            d3_buf   = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
            tgt_inv  = mean_demand.get(p, demand_d1.get(p, 0)) * TARGET_DAYS_INV
            # Skip in best-case scenario if no real need
            if scenario == 3 and inv_now >= tgt_inv and d3_buf == 0:
                continue
            candidates.append((p, d3_buf))

        candidates.sort(key=lambda x: -x[1])   # highest D+3 need first

        for p, d3_buf in candidates:
            last            = machine_last_part.get(m)
            changeover_flag = "No" if last == p else "Yes"
            r_val           = rate.get(p, 1)
            run_h           = max(MIN_RUN_HOURS, remaining)
            run_h           = min(run_h, remaining)
            qty             = round(run_h * r_val, 0)

            machine_hours[m]       += run_h
            current_inventory[p]    = current_inventory.get(p, 0) + qty
            machine_last_part[m]    = p

            filler.append({
                "Part":           p,
                "Machine":        m,
                "Run_Hours":      round(run_h, 2),
                "Production_Qty": qty,
                "Type":           "New — D3 pre-build" if d3_buf > 0 else "New — inv fill",
                "Changeover":     changeover_flag,
                "D1_Demand_15th": round(demand_d1.get(p, 0), 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            })
            machine_last_part[m] = p
            break

    return filler

# =============================================================
# SECTION 15 — MAIN SCHEDULER
# =============================================================

def schedule(parts, compatibility, machines, label=""):

    print(f"\n{'─'*62}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(machines)} machines")
    print(f"{'─'*62}")

    # Scenario
    scenario, _, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    # Demand horizon
    horizon_df = compute_demand_horizon(parts, compatibility, machines)

    pre_build_parts = horizon_df[horizon_df["D3_Pre_Build_Buffer"] > 0]
    if not pre_build_parts.empty:
        print(f"\n  D+3 pre-build required ({len(pre_build_parts)} parts):")
        for _, r in pre_build_parts.iterrows():
            print(f"    {r['Part']:30s}  "
                  f"15th={r['D1_Actual_15th']:.0f}  "
                  f"17th_tent={r['D3_Tentative_17th']:.0f}  "
                  f"17th_risk={r['D3_Risk_Adjusted']:.0f}  "
                  f"pre_build={r['D3_Pre_Build_Buffer']:.0f}")
    else:
        print("  No D+3 pre-build needed — 16th+17th capacity covers 17th demand")

    # State
    machine_hours     = {m: 0.0 for m in machines}
    machine_last_part = {m: machine_state.get(m) for m in machines}
    current_inventory = inventory.copy()

    plan, deferred, not_planned = [], [], []

    # Priority sort
    priority_df = compute_priority(parts, horizon_df)
    horizon_idx = horizon_df.set_index("Part")

    print(f"\n  Priority order:")
    for _, r in priority_df.iterrows():
        print(f"    {r['Part']:30s}  "
              f"score={r['Score']:.3f}  "
              f"cov={r['Days_Coverage']}d  "
              f"D1_short={max(0, r['D1_Demand_15th'] - inventory.get(r['Part'], 0)):.0f}  "
              f"D3_prebuild={r['D3_Pre_Build']:.0f}")

    # ── Main assignment loop ────────────────────────────────
    print(f"\n  Assignments:")
    for _, row in priority_df.iterrows():
        part     = row["Part"]
        inv_now  = current_inventory.get(part, 0)
        d1       = demand_d1.get(part, 0)
        d3_buf   = row["D3_Pre_Build"]
        r_val    = rate.get(part, 1)

        # Can we defer? Only if inventory covers D+1 AND no D+3 buffer needed
        inv_covers_d1 = inv_now >= d1
        if inv_covers_d1 and d3_buf == 0 and scenario >= 2:
            deferred.append({
                "Part":              part,
                "Inventory_Now":     round(inv_now, 0),
                "D1_Demand_15th":    round(d1, 0),
                "Inventory_Surplus": round(inv_now - d1, 0),
                "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                "Days_Coverage":     round(inv_now / max(mean_demand.get(part, d1), 1), 2),
                "Reason":            "Inventory covers 15th demand — no D+3 pre-build needed",
                "Next_Action":       "Plan on 15th if inventory drops below 1-day coverage",
            })
            continue

        # Today's total target hours
        today_qty  = row["Today_Target"]
        target_hrs = max(MIN_RUN_HOURS, today_qty / r_val if r_val > 0 else MIN_RUN_HOURS)

        # Uber-style machine ranking
        ranked   = rank_machines(part, machines, compatibility,
                                 machine_hours, machine_last_part)
        assigned = False

        for m, cost, free_h, co_penalty in ranked:
            run_h = min(target_hrs, free_h)
            run_h = max(run_h, MIN_RUN_HOURS)
            run_h = min(run_h, free_h)   # never exceed available
            qty   = round(run_h * r_val, 0)

            machine_hours[m]        += run_h
            current_inventory[part]  = current_inventory.get(part, 0) + qty
            machine_last_part[m]     = part

            plan.append({
                "Part":           part,
                "Machine":        m,
                "Run_Hours":      round(run_h, 2),
                "Production_Qty": qty,
                "D1_Demand_15th": round(d1, 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(part, 0), 0),
                "D3_Pre_Build":   round(d3_buf, 0),
                "Changeover":     "No" if co_penalty == 0 else "Yes",
                "Type":           "Primary",
                "Priority_Score": row["Score"],
            })

            co_str = "No " if co_penalty == 0 else "Yes"
            print(f"    ✓ {part:30s} → {m:15s}  "
                  f"{run_h:.2f}h  qty={qty:>8.0f}  "
                  f"CO={co_str}  "
                  f"[D1={d1:.0f} | D3buf={d3_buf:.0f}]")
            assigned = True
            break

        if not assigned:
            # Try splitting across multiple compatible machines
            remaining_target = target_hrs
            split_done = False

            for m, cost, free_h, co_penalty in ranked:
                if free_h < MIN_RUN_HOURS or remaining_target <= 0:
                    break
                run_h = min(remaining_target, free_h)
                run_h = max(run_h, MIN_RUN_HOURS)
                qty   = round(run_h * r_val, 0)

                machine_hours[m]        += run_h
                current_inventory[part]  = current_inventory.get(part, 0) + qty
                machine_last_part[m]     = part
                remaining_target        -= run_h

                plan.append({
                    "Part":           part,
                    "Machine":        m,
                    "Run_Hours":      round(run_h, 2),
                    "Production_Qty": qty,
                    "D1_Demand_15th": round(d1, 0),
                    "D3_Tent_17th":   round(demand_d3_tent.get(part, 0), 0),
                    "D3_Pre_Build":   round(d3_buf, 0),
                    "Changeover":     "No" if co_penalty == 0 else "Yes",
                    "Type":           "Split",
                    "Priority_Score": row["Score"],
                })
                print(f"    ↔ {part:30s} → {m:15s}  "
                      f"{run_h:.2f}h  [SPLIT]")
                split_done = True

            if not split_done:
                not_planned.append({
                    "Part":              part,
                    "D1_Demand_15th":    round(d1, 0),
                    "Inventory_Now":     round(inv_now, 0),
                    "Shortage":          round(max(0, d1 - inv_now), 0),
                    "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                    "Compatible_Machines": ", ".join(compatibility.get(part, [])) or "NONE",
                    "Reason":            "No compatible machine with sufficient free hours",
                    "Action_Needed":     "Review machine capacity or reschedule lower-priority parts",
                })
                print(f"    ✗ {part:30s}  NOT PLANNED — no capacity")

    # 22-hour filler
    print(f"\n  22-hour filler:")
    filler_rows = fill_remaining_hours(
        plan, machine_hours, machine_last_part,
        list(parts), compatibility, machines,
        current_inventory, horizon_df, scenario
    )
    if filler_rows:
        for fr in filler_rows:
            plan.append(fr)
            print(f"    + {fr['Part']:30s} → {fr['Machine']:15s}  "
                  f"{fr['Run_Hours']}h  [{fr['Type']}]  CO={fr['Changeover']}")
    else:
        print("    All machines fully utilised — no filler needed")

    # ── Inventory health after today's plan ─────────────────
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        d1       = demand_d1.get(p, 0)
        d3_t     = demand_d3_tent.get(p, 0)
        produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)

        # final_inventory = inventory + produced − demand_supplied
        inv_after = inv_b + produced - d1
        mean      = mean_demand.get(p, max(d1, 1))
        days_cov  = inv_after / mean if mean > 0 else 0

        # How much of 17th tentative is covered by this inventory?
        d3_covered  = min(inv_after, d3_t)
        d3_still_gap = max(0, d3_t - inv_after)

        inv_rows.append({
            "Part":                   p,
            "Inv_Before":             round(inv_b, 0),
            "Produced_14th":          round(produced, 0),
            "D1_Supplied_15th":       round(d1, 0),
            "Inv_After_15th_Supply":  round(inv_after, 0),   # = inv + produced - d1
            "Days_Coverage":          round(days_cov, 2),
            "D3_Tentative_17th":      round(d3_t, 0),
            "D3_Covered_By_Inv":      round(d3_covered, 0),
            "D3_Still_Needs_16th_17th": round(d3_still_gap, 0),
            "Status":                 ("OK"       if days_cov >= 1
                                       else "CRITICAL" if inv_after < 0
                                       else "LOW"),
        })

    # Machine utilization — rich columns including parts list
    mach_rows = []
    for m in machines:
        used       = machine_hours.get(m, 0)
        remaining  = AVAILABLE_HOURS - used
        parts_run  = [r["Part"] for r in plan if r["Machine"] == m]
        parts_str  = ", ".join(parts_run) if parts_run else "— idle —"
        co_count   = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")

        if used >= AVAILABLE_HOURS - 0.5:
            util_status = "FULL"
        elif used >= AVAILABLE_HOURS * 0.85:
            util_status = "GOOD"
        elif used >= AVAILABLE_HOURS * 0.5:
            util_status = "PARTIAL"
        else:
            util_status = "UNDERUSED"

        mach_rows.append({
            "Machine":           m,
            "Total_Available_Hrs": AVAILABLE_HOURS,
            "Used_Hours":        round(used, 2),
            "Unused_Hours":      round(remaining, 2),
            "Utilization_%":     round(used / AVAILABLE_HOURS * 100, 1),
            "Status":            util_status,
            "Parts_Planned":     len(parts_run),
            "Changeovers":       co_count,
            "Last_Part_Run":     machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": parts_str,
        })

    plan_df     = pd.DataFrame(plan)        if plan        else pd.DataFrame()
    def_df      = pd.DataFrame(deferred)    if deferred    else pd.DataFrame()
    not_df      = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df     = pd.DataFrame(mach_rows)
    inv_df      = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)

    return plan_df, def_df, not_df, mach_df, inv_df, machine_last_part, horizon_df

# =============================================================
# SECTION 16 — RUN BOTH MACHINE GROUPS
# =============================================================

hz_parts = data[data["Material"].isin(hz_matrix["Part"])]["Material"].unique()
vt_parts = data[data["Material"].isin(vt_matrix["Part"])]["Material"].unique()

hz_plan, hz_def, hz_not, hz_mach, hz_inv, hz_state, hz_horizon = \
    schedule(hz_parts, hz_compat, hz_machines, "HZ Machines")

vt_plan, vt_def, vt_not, vt_mach, vt_inv, vt_state, vt_horizon = \
    schedule(vt_parts, vt_compat, vt_machines, "VT Machines")

save_machine_state(hz_state, vt_state)

# =============================================================
# SECTION 17 — SAVE OUTPUT
# =============================================================

# =============================================================
# SECTION 17 — SAVE OUTPUT  (formatted Excel)
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Colour palette for header rows
HEADER_COLORS = {
    "HZ_Plan":               "1F4E79",   # dark blue
    "VT_Plan":               "1F4E79",
    "HZ_Machine_Util":       "375623",   # dark green
    "VT_Machine_Util":       "375623",
    "HZ_Not_Planned":        "7B2C2C",   # dark red
    "VT_Not_Planned":        "7B2C2C",
    "HZ_Deferred":           "7F6000",   # dark amber
    "VT_Deferred":           "7F6000",
    "HZ_Inventory_Health":   "4A235A",   # dark purple
    "VT_Inventory_Health":   "4A235A",
    "HZ_Demand_Horizon":     "154360",
    "VT_Demand_Horizon":     "154360",
    "ALL_Plan_Combined":     "1F4E79",
}

STATUS_FILLS = {
    "FULL":      PatternFill("solid", fgColor="C6EFCE"),   # green
    "GOOD":      PatternFill("solid", fgColor="DDEBF7"),   # blue
    "PARTIAL":   PatternFill("solid", fgColor="FFEB9C"),   # yellow
    "UNDERUSED": PatternFill("solid", fgColor="FFC7CE"),   # red
    "OK":        PatternFill("solid", fgColor="C6EFCE"),
    "LOW":       PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":  PatternFill("solid", fgColor="FFC7CE"),
    "PRE-BUILD NEEDED": PatternFill("solid", fgColor="FFC7CE"),
    "WATCH — near capacity": PatternFill("solid", fgColor="FFEB9C"),
}

def style_sheet(ws, header_hex):
    """Apply header formatting and auto column widths to a worksheet."""
    header_fill = PatternFill("solid", fgColor=header_hex)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    center      = Alignment(horizontal="center", vertical="center", wrap_text=True)

    # Style header row
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center

    ws.row_dimensions[1].height = 32

    # Auto-fit column widths based on content
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                cell_len = len(str(cell.value)) if cell.value is not None else 0
                max_len  = max(max_len, cell_len)
            except Exception:
                pass
        # Cap between 10 and 50 characters wide
        ws.column_dimensions[col_letter].width = max(10, min(50, max_len + 3))

    # Colour status/flag cells
    status_col_names = {"Status", "D3_Flag", "Utilization_Status"}
    header_row = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(header_row, start=1):
        if col_name in status_col_names or "Status" in str(col_name):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value), None)
                    if fill:
                        cell.fill = fill

    # Freeze the header row
    ws.freeze_panes = "A2"

print(f"\nWriting → {output_path}")

all_plan = pd.concat([hz_plan, vt_plan], ignore_index=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # ── Plans ────────────────────────────────────────────────
    hz_plan.to_excel(writer, sheet_name="HZ_Plan",           index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan",           index=False)

    # ── Machine utilization ──────────────────────────────────
    hz_mach.to_excel(writer, sheet_name="HZ_Machine_Util",   index=False)
    vt_mach.to_excel(writer, sheet_name="VT_Machine_Util",   index=False)

    # ── Not planned (capacity exhausted) ────────────────────
    hz_not.to_excel(writer,  sheet_name="HZ_Not_Planned",    index=False)
    vt_not.to_excel(writer,  sheet_name="VT_Not_Planned",    index=False)

    # ── Deferred (inventory sufficient) ─────────────────────
    hz_def.to_excel(writer,  sheet_name="HZ_Deferred",       index=False)
    vt_def.to_excel(writer,  sheet_name="VT_Deferred",       index=False)

    # ── Inventory health ─────────────────────────────────────
    hz_inv.to_excel(writer,  sheet_name="HZ_Inventory_Health", index=False)
    vt_inv.to_excel(writer,  sheet_name="VT_Inventory_Health", index=False)

    # ── Demand horizon ───────────────────────────────────────
    hz_horizon.to_excel(writer, sheet_name="HZ_Demand_Horizon", index=False)
    vt_horizon.to_excel(writer, sheet_name="VT_Demand_Horizon", index=False)

    # ── Combined plan ────────────────────────────────────────
    all_plan.to_excel(writer, sheet_name="ALL_Plan_Combined", index=False)

# Apply formatting after writing (openpyxl post-process)
wb = load_workbook(output_path)
for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames:
        style_sheet(wb[sheet_name], header_hex)

# Set sheet tab order — most important sheets first
tab_order = [
    "HZ_Plan", "VT_Plan",
    "HZ_Machine_Util", "VT_Machine_Util",
    "HZ_Not_Planned", "VT_Not_Planned",
    "HZ_Deferred", "VT_Deferred",
    "HZ_Inventory_Health", "VT_Inventory_Health",
    "HZ_Demand_Horizon", "VT_Demand_Horizon",
    "ALL_Plan_Combined",
]
for i, name in enumerate(tab_order):
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = (
            "1F4E79" if "Plan" in name else
            "375623" if "Machine" in name else
            "7B2C2C" if "Not_Planned" in name else
            "7F6000" if "Deferred" in name else
            "4A235A"
        )

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 18 — SUMMARY PRINT
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V4 Complete  —  {PLANNING_DATE}")
print(f"{'='*62}")
print(f"  HZ  planned={len(hz_plan):>4}  deferred={len(hz_def):>4}  not_planned={len(hz_not):>4}")
print(f"  VT  planned={len(vt_plan):>4}  deferred={len(vt_def):>4}  not_planned={len(vt_not):>4}")

all_horizon = pd.concat([hz_horizon, vt_horizon], ignore_index=True)
pre_build   = all_horizon[all_horizon["D3_Pre_Build_Buffer"] > 0]

if not pre_build.empty:
    print(f"\n  D+3 (17th) pre-build buffers applied to {len(pre_build)} parts:")
    print(f"  {'Part':<30}  {'15th demand':>12}  {'17th tent':>10}  {'17th risk':>10}  {'buffer':>8}")
    print(f"  {'─'*30}  {'─'*12}  {'─'*10}  {'─'*10}  {'─'*8}")
    for _, r in pre_build.iterrows():
        print(f"  {r['Part']:<30}  {r['D1_Actual_15th']:>12.0f}  "
              f"{r['D3_Tentative_17th']:>10.0f}  "
              f"{r['D3_Risk_Adjusted']:>10.0f}  "
              f"{r['D3_Pre_Build_Buffer']:>8.0f}")
else:
    print(f"\n  No D+3 pre-build needed — 16th and 17th machine capacity is sufficient")

print(f"\n  Output  → {output_path}")
print(f"  State   → {MACHINE_STATE_FILE}")
print(f"\n  NOTE: Change the 3 lines in SECTION 1 each morning before running.")
print(f"        Tomorrow (15th): ACTUAL_DEMAND_COL    = '2026-03-13 Total Production Plan'")
print(f"                         TENTATIVE_DEMAND_COL = '2026-03-15 Total Production Plan'")
print(f"                         PLANNING_DATE        = date(2026, 3, 15)")


  Smart APS V4  —  Planning date: 2026-03-14
  Actual demand col   : 2026-03-12 Total Production Plan
  Tentative demand col: 2026-03-14 Total Production Plan
  16th March          : IGNORED today — will plan on 15th

Loading data...
Validating demand columns...
  Actual demand col        : '2026-03-12 Total Production Plan'  ✓
  Tentative demand col     : '2026-03-14 Total Production Plan'  ✓
  Machine state         : FIRST RUN — no previous state
                          No changeover penalties applied today.
                          State will be saved to 'machine_state.json'
                          after this run for use tomorrow.

──────────────────────────────────────────────────────────────
  HZ Machines  |  274 parts  |  29 machines
──────────────────────────────────────────────────────────────
  SCENARIO 1 — 190/274 parts critical

  D+3 pre-build required (4 parts):
    14MA410307-00003X2              15th=680  17th_tent=680  17th_risk=680  pre_build=524
    S03066-003A0

In [4]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, date
 
# =============================================================
# ██████╗  █████╗ ██████╗  █████╗ ███╗   ███╗███████╗
# ██╔══██╗██╔══██╗██╔══██╗██╔══██╗████╗ ████║██╔════╝
# ██████╔╝███████║██████╔╝███████║██╔████╔██║███████╗
# ██╔═══╝ ██╔══██║██╔══██╗██╔══██║██║╚██╔╝██║╚════██║
# ██║     ██║  ██║██║  ██║██║  ██║██║ ╚═╝ ██║███████║
# ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═╝╚═╝     ╚═╝╚══════╝
#
#  Smart APS V4  —  Forward-Looking Demand Horizon
#  Planning date  : 14th March 2026
#  Actual col     : "2026-03-12 Total Production Plan"  (supply on 15th)
#  Tentative col  : "2026-03-14 Total Production Plan"  (pre-build for 17th)
#  16th March     : ignored today, planned on 15th when data arrives
# =============================================================
 
# =============================================================
# SECTION 1 — COLUMN NAMES  (change these every day)
# -------------------------------------------------------------
# On 14th March your demand file has:
#   "2026-03-12 Total Production Plan" → actual for 15th  (D+1)
#   "2026-03-14 Total Production Plan" → tentative for 17th (D+3)
#
# Every morning update the two column name strings below
# to match whatever columns appear in that day's file.
# The planning date is only used for naming the output file.
# =============================================================
 
ACTUAL_DEMAND_COL    = "2026-03-12 Total Production Plan"   # D+1  ← change daily
TENTATIVE_DEMAND_COL = "2026-03-14 Total Production Plan"   # D+3  ← change daily
PLANNING_DATE        = date(2026, 3, 11)                    # today ← change daily
 
# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================
 
AVAILABLE_HOURS    = 22       # total machine hours per shift
MIN_RUN_HOURS      = 4        # minimum block per part on a machine
TARGET_DAYS_INV    = 3        # ideal inventory buffer (days)
MACHINE_STATE_FILE = "machine_state.json"
 
# Z-score for tentative demand buffer
# Since your tentative is nearly accurate → Z = 1.28 (90% coverage, modest buffer)
# If tentative were very volatile we'd use 1.65 or 2.0
Z_FOR_TENTATIVE    = 1.28
 
# Priority weights
W_COVERAGE   = 2.0
W_URGENCY    = 3.0
W_RISK       = 1.5
W_D3_BOOST   = 2.0   # how strongly D+3 pre-build need lifts priority
 
# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================
 
book_path   = "C:/Users/Ex0164/Book1.xlsx"
daily_path  = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
output_path = f"Smart_APS_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"
 
# =============================================================
# SECTION 4 — LOAD DATA
# =============================================================
 
print(f"\n{'='*62}")
print(f"  Smart APS V4  —  Planning date: {PLANNING_DATE}")
print(f"  Actual demand col   : {ACTUAL_DEMAND_COL}")
print(f"  Tentative demand col: {TENTATIVE_DEMAND_COL}")
print(f"  16th March          : IGNORED today — will plan on 15th")
print(f"{'='*62}\n")
 
print("Loading data...")
stats     = pd.read_excel(book_path,   sheet_name="Sheet2")
daily     = pd.read_excel(daily_path,  sheet_name="Sheet1")
hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")
 
# =============================================================
# SECTION 5 — VALIDATE DEMAND COLUMNS
# -------------------------------------------------------------
# Columns are read directly by name — no date parsing needed.
# If a column is missing, a clear error is raised showing what
# columns are actually in the file so you know what to fix.
# =============================================================
 
def validate_column(df, col_name, label):
    """Check the column exists; if not, print all columns and raise."""
    if col_name in df.columns:
        print(f"  {label:25s}: '{col_name}'  ✓")
        return
    raise ValueError(
        f"\n  ERROR: Column not found — {col_name}\n"
        f"  Label : {label}\n"
        f"  Fix   : Update {label.split()[0].upper()}_DEMAND_COL in Section 1\n"
        f"  Columns available in file:\n"
        + "\n".join(f"    '{c}'" for c in df.columns)
    )
 
print("Validating demand columns...")
validate_column(daily, ACTUAL_DEMAND_COL,    "Actual demand col")
validate_column(daily, TENTATIVE_DEMAND_COL, "Tentative demand col")
 
# =============================================================
# SECTION 6 — MERGE & PRODUCTION RATE
# =============================================================
 
data = stats.merge(daily, left_on="Part", right_on="Material")
 
data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]
data = data[data["Rate"].notna()].copy()
 
# =============================================================
# SECTION 7 — BUILD LOOKUP DICTIONARIES
# =============================================================
 
def safe_dict(df, key_col, val_col, default=0.0):
    return {
        k: (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
    }
 
inventory       = safe_dict(data, "Material", "Inventory on 24th")
rate            = safe_dict(data, "Material", "Rate")
 
# D+1: actual demand — must supply this (15th March)
demand_d1       = safe_dict(data, "Material", ACTUAL_DEMAND_COL)
 
# D+3: tentative demand — pre-build buffer for this (17th March)
demand_d3_tent  = safe_dict(data, "Material", TENTATIVE_DEMAND_COL)
 
# Historical stats (for smarter Z-score if available)
mean_demand = (safe_dict(data, "Material", "Mean_Demand")
               if "Mean_Demand" in data.columns else demand_d1.copy())
std_demand  = (safe_dict(data, "Material", "Std_Dev_Demand")
               if "Std_Dev_Demand" in data.columns
               else {k: 0.0 for k in demand_d1})
 
# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# =============================================================
 
# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# -------------------------------------------------------------
# This file is created automatically after the first run.
# On first run (file does not exist yet):
#   → No changeover penalty applied to anyone — clean slate
#   → After planning, the file is written with today's last parts
# From second run onwards:
#   → File is read, changeover penalties apply correctly
#   → Parts that ran last on a machine get zero changeover cost
# =============================================================
 
def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"  Machine state loaded  :")
        for m, p in state.items():
            print(f"    {m:25s} last ran → {p}")
        return state
    # First run — no state file exists yet
    print("  Machine state         : FIRST RUN — no previous state")
    print("                          No changeover penalties applied today.")
    print(f"                          State will be saved to '{MACHINE_STATE_FILE}'")
    print("                          after this run for use tomorrow.")
    return {}   # empty dict → machine_last_part[m] = None → no penalty
 
def save_machine_state(hz_state, vt_state):
    combined = {**hz_state, **vt_state}
    # Remove machines where last part is None (never ran)
    combined = {m: p for m, p in combined.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → {MACHINE_STATE_FILE}")
    print("  (Tomorrow's plan will use this to avoid unnecessary changeovers)")
    for m, p in combined.items():
        print(f"    {m:25s} last ran → {p}")
 
machine_state = load_machine_state()
 
# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================
 
def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines
 
hz_compat, hz_machines = build_compatibility(hz_matrix)
vt_compat, vt_machines = build_compatibility(vt_matrix)
 
# =============================================================
# SECTION 10 — DEMAND HORIZON ANALYSIS
# -------------------------------------------------------------
# For each part, calculate:
#
#   d1_shortage   = max(0, D+1 demand − current inventory)
#                 = what we MUST produce today for 15th supply
#
#   d3_buffer     = extra to pre-build today for 17th demand
#                 = max(0, risk_adjusted_D3 − inv_at_d3_start
#                           − machine_capacity_on_16th
#                           − machine_capacity_on_17th)
#
#   today_target  = d1_shortage + d3_buffer
#
# Why do we subtract capacity on 16th AND 17th?
#   Because even without a demand file for 16th, we can still
#   PRODUCE on 16th. So the pre-build needed today is only what
#   BOTH 16th and 17th combined cannot cover.
#
# Note: Since tentative is nearly accurate, Z = 1.28 (modest buffer)
# =============================================================
 
def compute_demand_horizon(parts, compatibility, machines):
 
    def max_daily_capacity(part):
        """Max units producible in one full day across all compatible machines."""
        compat_machines = compatibility.get(part, [])
        if not compat_machines:
            return 0
        r = rate.get(part, 1)
        # Each machine can run AVAILABLE_HOURS independently
        return len(compat_machines) * AVAILABLE_HOURS * r
 
    rows = []
    for p in parts:
        inv      = inventory.get(p, 0)
        d1       = demand_d1.get(p, 0)
        d3_t     = demand_d3_tent.get(p, 0)
        mean     = mean_demand.get(p, max(d1, 1))
        std      = std_demand.get(p, 0)
        r        = rate.get(p, 1)
 
        # ── D+1 (15th) shortage ─────────────────────────────
        d1_shortage    = max(0.0, d1 - inv)
        inv_after_d1   = max(0.0, inv - d1)   # inventory left after supplying 15th
 
        # ── D+3 (17th) risk-adjusted target ─────────────────
        # Since tentative is nearly accurate, small Z = 1.28
        d3_risk_adj    = d3_t + Z_FOR_TENTATIVE * std
 
        # ── Capacity available on 16th and 17th ─────────────
        # 16th: no demand file, machines run freely → full capacity available
        # 17th: machines run freely → full capacity available
        # Together they can cover: 2 × daily_capacity
        cap_16th       = max_daily_capacity(p)
        cap_17th       = max_daily_capacity(p)
 
        # Inventory at start of 17th =
        #   inv_after_d1 (left after 15th supply)
        #   + whatever we produce on 16th (up to cap_16th, but only if needed)
        # We assume 16th production goes toward 17th demand as well
        # So total coverage for 17th = inv_after_d1 + cap_16th + cap_17th
 
        total_coverage_for_d3 = inv_after_d1 + cap_16th + cap_17th
 
        # Pre-build needed today = gap that even 16th+17th machines can't cover
        d3_buffer = max(0.0, d3_risk_adj - total_coverage_for_d3)
 
        # ── Combined today's target ──────────────────────────
        today_target = d1_shortage + d3_buffer
 
        # ── D+3 status flag ─────────────────────────────────
        if d3_buffer > 0:
            d3_flag = "PRE-BUILD NEEDED"
        elif d3_risk_adj > (cap_16th + cap_17th) * 0.8:
            d3_flag = "WATCH — near capacity"
        else:
            d3_flag = "OK"
 
        rows.append({
            "Part":                   p,
            "Inventory_Now":          round(inv, 0),
 
            # D+1 (15th) columns
            "D1_Actual_15th":         round(d1, 0),
            "D1_Shortage":            round(d1_shortage, 0),
            "Inv_After_15th_Supply":  round(inv_after_d1, 0),
 
            # D+3 (17th) columns
            "D3_Tentative_17th":      round(d3_t, 0),
            "D3_Risk_Adjusted":       round(d3_risk_adj, 0),
            "Cap_16th_Available":     round(cap_16th, 0),
            "Cap_17th_Available":     round(cap_17th, 0),
            "D3_Pre_Build_Buffer":    round(d3_buffer, 0),
            "D3_Flag":                d3_flag,
 
            # Combined
            "Today_Total_Target_Qty": round(today_target, 0),
            "Today_Total_Target_Hrs": round(today_target / r if r > 0 else 0, 2),
        })
 
    return pd.DataFrame(rows)
 
# =============================================================
# SECTION 11 — SCENARIO CLASSIFIER
# =============================================================
 
def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv  = inventory.get(p, 0)
        mean = mean_demand.get(p, demand_d1.get(p, 1))
        coverage.append(inv / mean if mean > 0 else 999)
 
    n         = len(parts)
    critical  = sum(1 for c in coverage if c < 1)
    low       = sum(1 for c in coverage if c < TARGET_DAYS_INV)
 
    if critical == n:
        return 0, 2.00, "SCENARIO 0 — ALL parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, 1.65, f"SCENARIO 1 — {critical}/{n} parts critical"
    elif low > 0:
        return 2, 1.28, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, 1.28, f"SCENARIO 3 — All parts healthy (≥{TARGET_DAYS_INV} days)"
 
# =============================================================
# SECTION 12 — PRIORITY SCORING  (D+3-aware)
# =============================================================
 
def compute_priority(parts, horizon_df):
    horizon = horizon_df.set_index("Part")
    rows    = []
 
    for p in parts:
        inv  = inventory.get(p, 0)
        d1   = demand_d1.get(p, 0)
        mean = mean_demand.get(p, max(d1, 1))
        std  = std_demand.get(p, 0)
 
        days_cov = inv / mean if mean > 0 else 999
 
        # Urgency based on D+1 coverage
        if days_cov < 1:
            urgency = 1.0
        elif days_cov < 2:
            urgency = 0.5
        else:
            urgency = 0.0
 
        cv = std / mean if mean > 0 else 0
 
        # D+3 boost: if pre-build buffer > 0, add extra urgency
        d3_buf  = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
        d3_boost = min(1.0, d3_buf / max(mean, 1))
 
        score = (W_COVERAGE * (1.0 / (days_cov + 0.01))
               + W_URGENCY  * urgency
               + W_RISK     * cv
               + W_D3_BOOST * d3_boost)
 
        today_target = (horizon.loc[p, "Today_Total_Target_Qty"]
                        if p in horizon.index else d1)
 
        rows.append({
            "Part":           p,
            "Days_Coverage":  round(days_cov, 2),
            "Urgency_D1":     urgency,
            "CV":             round(cv, 3),
            "D3_Boost":       round(d3_boost, 3),
            "Score":          round(score, 4),
            "D1_Demand_15th": round(d1, 0),
            "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            "D3_Pre_Build":   round(d3_buf, 0),
            "Today_Target":   round(today_target, 0),
        })
 
    return (pd.DataFrame(rows)
              .sort_values("Score", ascending=False)
              .reset_index(drop=True))
 
# =============================================================
# SECTION 13 — UBER-STYLE MACHINE RANKER
# =============================================================
 
def rank_machines(part, machines, compatibility, machine_hours, machine_last_part):
    """
    Returns machines sorted by assignment cost (lowest = best choice).
    Cost = changeover_penalty + load_factor
    Same part ran last on machine → 0 changeover penalty (zero cost bonus)
    """
    ranked = []
    for m in machines:
        if m not in compatibility.get(part, []):
            continue
        used = machine_hours.get(m, 0)
        free = AVAILABLE_HOURS - used
        if free < MIN_RUN_HOURS:
            continue
        last               = machine_last_part.get(m)
        changeover_penalty = 0.0 if last == part else 0.5
        cost               = changeover_penalty + (used / AVAILABLE_HOURS)
        ranked.append((m, cost, free, changeover_penalty))
 
    ranked.sort(key=lambda x: x[1])
    return ranked
 
# =============================================================
# SECTION 14 — 22-HOUR UTILIZATION FILLER
# =============================================================
 
def fill_remaining_hours(plan, machine_hours, machine_last_part,
                         parts, compatibility, machines,
                         current_inventory, horizon_df, scenario):
    """
    After primary assignments, fill remaining machine time.
    Priority order:
      1. Extend an already-planned part (zero changeover) if it still
         needs more production (D+3 buffer not yet met OR below 3-day inv)
      2. Add a new part — pick the one with the highest D+3 pre-build need
    """
    filler = []
    horizon = horizon_df.set_index("Part")
 
    for m in machines:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
        if remaining < MIN_RUN_HOURS:
            continue
 
        on_machine = [row["Part"] for row in plan if row["Machine"] == m]
 
        # ── Option A: extend already-planned part ────────────
        extended = False
        for p in on_machine:
            inv_now    = current_inventory.get(p, 0)
            target_inv = mean_demand.get(p, demand_d1.get(p, 0)) * TARGET_DAYS_INV
            d3_buf     = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
 
            if inv_now < target_inv or d3_buf > 0:
                r_val     = rate.get(p, 1)
                extra_qty = round(remaining * r_val, 0)
                machine_hours[m]       += remaining
                current_inventory[p]    = current_inventory.get(p, 0) + extra_qty
 
                filler.append({
                    "Part":           p,
                    "Machine":        m,
                    "Run_Hours":      round(remaining, 2),
                    "Production_Qty": extra_qty,
                    "Type":           "Extend — D3 buffer" if d3_buf > 0 else "Extend — inv build",
                    "Changeover":     "No",
                    "D1_Demand_15th": round(demand_d1.get(p, 0), 0),
                    "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
                })
                extended = True
                break
 
        if extended:
            continue
 
        # ── Option B: new part — highest D+3 need first ──────
        candidates = []
        for p in parts:
            if p in on_machine:
                continue
            if m not in compatibility.get(p, []):
                continue
            inv_now  = current_inventory.get(p, 0)
            d3_buf   = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
            tgt_inv  = mean_demand.get(p, demand_d1.get(p, 0)) * TARGET_DAYS_INV
            # Skip in best-case scenario if no real need
            if scenario == 3 and inv_now >= tgt_inv and d3_buf == 0:
                continue
            candidates.append((p, d3_buf))
 
        candidates.sort(key=lambda x: -x[1])   # highest D+3 need first
 
        for p, d3_buf in candidates:
            last            = machine_last_part.get(m)
            changeover_flag = "No" if last == p else "Yes"
            r_val           = rate.get(p, 1)
            run_h           = max(MIN_RUN_HOURS, remaining)
            run_h           = min(run_h, remaining)
            qty             = round(run_h * r_val, 0)
 
            machine_hours[m]       += run_h
            current_inventory[p]    = current_inventory.get(p, 0) + qty
            machine_last_part[m]    = p
 
            filler.append({
                "Part":           p,
                "Machine":        m,
                "Run_Hours":      round(run_h, 2),
                "Production_Qty": qty,
                "Type":           "New — D3 pre-build" if d3_buf > 0 else "New — inv fill",
                "Changeover":     changeover_flag,
                "D1_Demand_15th": round(demand_d1.get(p, 0), 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            })
            machine_last_part[m] = p
            break
 
    return filler
 
# =============================================================
# SECTION 15 — MAIN SCHEDULER
# =============================================================
 
def schedule(parts, compatibility, machines, label=""):
 
    print(f"\n{'─'*62}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(machines)} machines")
    print(f"{'─'*62}")
 
    # Scenario
    scenario, _, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
 
    # Demand horizon
    horizon_df = compute_demand_horizon(parts, compatibility, machines)
 
    pre_build_parts = horizon_df[horizon_df["D3_Pre_Build_Buffer"] > 0]
    if not pre_build_parts.empty:
        print(f"\n  D+3 pre-build required ({len(pre_build_parts)} parts):")
        for _, r in pre_build_parts.iterrows():
            print(f"    {r['Part']:30s}  "
                  f"15th={r['D1_Actual_15th']:.0f}  "
                  f"17th_tent={r['D3_Tentative_17th']:.0f}  "
                  f"17th_risk={r['D3_Risk_Adjusted']:.0f}  "
                  f"pre_build={r['D3_Pre_Build_Buffer']:.0f}")
    else:
        print("  No D+3 pre-build needed — 16th+17th capacity covers 17th demand")
 
    # State
    machine_hours     = {m: 0.0 for m in machines}
    machine_last_part = {m: machine_state.get(m) for m in machines}
    current_inventory = inventory.copy()
 
    plan, deferred, not_planned = [], [], []
 
    # Priority sort
    priority_df = compute_priority(parts, horizon_df)
    horizon_idx = horizon_df.set_index("Part")
 
    print(f"\n  Priority order:")
    for _, r in priority_df.iterrows():
        print(f"    {r['Part']:30s}  "
              f"score={r['Score']:.3f}  "
              f"cov={r['Days_Coverage']}d  "
              f"D1_short={max(0, r['D1_Demand_15th'] - inventory.get(r['Part'], 0)):.0f}  "
              f"D3_prebuild={r['D3_Pre_Build']:.0f}")
 
    # ── Main assignment loop ────────────────────────────────
    print(f"\n  Assignments:")
    for _, row in priority_df.iterrows():
        part     = row["Part"]
        inv_now  = current_inventory.get(part, 0)
        d1       = demand_d1.get(part, 0)
        d3_buf   = row["D3_Pre_Build"]
        r_val    = rate.get(part, 1)
 
        inv_covers_d1  = inv_now >= d1
        compatible_mch = compatibility.get(part, [])
        days_cov_now   = inv_now / max(mean_demand.get(part, d1), 1)
 
        # ── Decision 1: part is not needed today ───────────────
        # Conditions: inventory already covers D+1 demand
        #             AND no D+3 pre-build buffer is required
        #             AND scenario is not critical (0 or 1)
        # → goes to "Not Required Today" list — this is GOOD, not a failure
        if inv_covers_d1 and d3_buf == 0 and scenario >= 2:
            deferred.append({
                "Part":              part,
                "Inventory_Now":     round(inv_now, 0),
                "D1_Demand_15th":    round(d1, 0),
                "Inventory_Surplus": round(inv_now - d1, 0),
                "Days_Coverage":     round(days_cov_now, 2),
                "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                "Reason":            "Inventory sufficient — covers D+1 demand with surplus",
                "Next_Action":       "Will be re-evaluated tomorrow",
            })
            print(f"    - {part:30s}  NOT REQUIRED TODAY"
                  f"  inv={inv_now:.0f}  d1={d1:.0f}  surplus={inv_now-d1:.0f}")
            continue
 
        # ── Decision 2: part needs production — find a machine ─
        today_qty  = row["Today_Target"]
        target_hrs = max(MIN_RUN_HOURS, today_qty / r_val if r_val > 0 else MIN_RUN_HOURS)
 
        # Check if part even has compatible machines defined
        if not compatible_mch:
            not_planned.append({
                "Part":                part,
                "D1_Demand_15th":      round(d1, 0),
                "Inventory_Now":       round(inv_now, 0),
                "Shortage":            round(max(0, d1 - inv_now), 0),
                "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                "Compatible_Machines": "NONE DEFINED",
                "Reason":              "Part has no compatible machines in the matrix",
                "Action_Needed":       "Add this part to the compatibility matrix",
            })
            print(f"    ✗ {part:30s}  NOT PLANNED — not in compatibility matrix")
            continue
 
        # Uber-style machine ranking
        ranked   = rank_machines(part, machines, compatibility,
                                 machine_hours, machine_last_part)
 
        # All compatible machines exist but none have free hours
        if not ranked:
            # Determine precise reason
            all_full    = all(machine_hours.get(m, 0) >= AVAILABLE_HOURS - MIN_RUN_HOURS
                              for m in compatible_mch if m in machine_hours)
            not_in_group = [m for m in compatible_mch if m not in machine_hours]
 
            if not_in_group:
                reason  = f"Compatible machines {not_in_group} not in this machine group"
                action  = "Check compatibility matrix — machine may belong to wrong group (HZ/VT)"
            elif all_full:
                reason  = f"All compatible machines fully utilised ({', '.join(compatible_mch)})"
                action  = "Increase machine hours, reduce qty of lower-priority parts, or plan tomorrow"
            else:
                reason  = "Compatible machines exist but remaining hours below MIN_RUN_HOURS (4h)"
                action  = f"Reduce MIN_RUN_HOURS or free up capacity on {', '.join(compatible_mch)}"
 
            not_planned.append({
                "Part":                part,
                "D1_Demand_15th":      round(d1, 0),
                "Inventory_Now":       round(inv_now, 0),
                "Shortage":            round(max(0, d1 - inv_now), 0),
                "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                "Compatible_Machines": ", ".join(compatible_mch),
                "Reason":              reason,
                "Action_Needed":       action,
            })
            print(f"    ✗ {part:30s}  NOT PLANNED — {reason}")
            continue
 
        assigned = False
 
        for m, cost, free_h, co_penalty in ranked:
            run_h = min(target_hrs, free_h)
            run_h = max(run_h, MIN_RUN_HOURS)
            run_h = min(run_h, free_h)
            qty   = round(run_h * r_val, 0)
 
            machine_hours[m]        += run_h
            current_inventory[part]  = current_inventory.get(part, 0) + qty
            machine_last_part[m]     = part
 
            plan.append({
                "Part":           part,
                "Machine":        m,
                "Run_Hours":      round(run_h, 2),
                "Production_Qty": qty,
                "D1_Demand_15th": round(d1, 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(part, 0), 0),
                "D3_Pre_Build":   round(d3_buf, 0),
                "Changeover":     "No" if co_penalty == 0 else "Yes",
                "Type":           "Primary",
                "Priority_Score": row["Score"],
            })
 
            co_str = "No " if co_penalty == 0 else "Yes"
            print(f"    ✓ {part:30s} → {m:15s}  "
                  f"{run_h:.2f}h  qty={qty:>8.0f}  "
                  f"CO={co_str}  "
                  f"[D1={d1:.0f} | D3buf={d3_buf:.0f}]")
            assigned = True
            break
 
        if not assigned:
            # Try splitting across multiple compatible machines
            remaining_target = target_hrs
            split_done       = False
 
            for m, cost, free_h, co_penalty in ranked:
                if free_h < MIN_RUN_HOURS or remaining_target <= 0:
                    break
                run_h = min(remaining_target, free_h)
                run_h = max(run_h, MIN_RUN_HOURS)
                qty   = round(run_h * r_val, 0)
 
                machine_hours[m]        += run_h
                current_inventory[part]  = current_inventory.get(part, 0) + qty
                machine_last_part[m]     = part
                remaining_target        -= run_h
 
                plan.append({
                    "Part":           part,
                    "Machine":        m,
                    "Run_Hours":      round(run_h, 2),
                    "Production_Qty": qty,
                    "D1_Demand_15th": round(d1, 0),
                    "D3_Tent_17th":   round(demand_d3_tent.get(part, 0), 0),
                    "D3_Pre_Build":   round(d3_buf, 0),
                    "Changeover":     "No" if co_penalty == 0 else "Yes",
                    "Type":           "Split",
                    "Priority_Score": row["Score"],
                })
                print(f"    ↔ {part:30s} → {m:15s}  "
                      f"{run_h:.2f}h  [SPLIT]")
                split_done = True
 
            if not split_done:
                # Ranked was non-empty but every machine was too full for even MIN_RUN_HOURS
                machines_status = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                                   for m in compatible_mch if m in machine_hours}
                reason = ("All compatible machines have less than MIN_RUN_HOURS "
                          f"({MIN_RUN_HOURS}h) remaining. "
                          f"Free hours: "
                          + ", ".join(f"{m}={h}h" for m, h in machines_status.items()))
 
                not_planned.append({
                    "Part":                part,
                    "D1_Demand_15th":      round(d1, 0),
                    "Inventory_Now":       round(inv_now, 0),
                    "Shortage":            round(max(0, d1 - inv_now), 0),
                    "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              reason,
                    "Action_Needed":       "Inventory will need to cover demand — check Inventory_Health sheet",
                })
                print(f"    ✗ {part:30s}  NOT PLANNED — capacity too fragmented")
 
    # 22-hour filler
    print(f"\n  22-hour filler:")
    filler_rows = fill_remaining_hours(
        plan, machine_hours, machine_last_part,
        list(parts), compatibility, machines,
        current_inventory, horizon_df, scenario
    )
    if filler_rows:
        for fr in filler_rows:
            plan.append(fr)
            print(f"    + {fr['Part']:30s} → {fr['Machine']:15s}  "
                  f"{fr['Run_Hours']}h  [{fr['Type']}]  CO={fr['Changeover']}")
    else:
        print("    All machines fully utilised — no filler needed")
 
    # ── Inventory health after today's plan ─────────────────
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        d1       = demand_d1.get(p, 0)
        d3_t     = demand_d3_tent.get(p, 0)
        produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)
 
        # final_inventory = inventory + produced − demand_supplied
        inv_after = inv_b + produced - d1
        mean      = mean_demand.get(p, max(d1, 1))
        days_cov  = inv_after / mean if mean > 0 else 0
 
        # How much of 17th tentative is covered by this inventory?
        d3_covered  = min(inv_after, d3_t)
        d3_still_gap = max(0, d3_t - inv_after)
 
        inv_rows.append({
            "Part":                   p,
            "Inv_Before":             round(inv_b, 0),
            "Produced_14th":          round(produced, 0),
            "D1_Supplied_15th":       round(d1, 0),
            "Inv_After_15th_Supply":  round(inv_after, 0),   # = inv + produced - d1
            "Days_Coverage":          round(days_cov, 2),
            "D3_Tentative_17th":      round(d3_t, 0),
            "D3_Covered_By_Inv":      round(d3_covered, 0),
            "D3_Still_Needs_16th_17th": round(d3_still_gap, 0),
            "Status":                 ("OK"       if days_cov >= 1
                                       else "CRITICAL" if inv_after < 0
                                       else "LOW"),
        })
 
    # Machine utilization — rich columns including parts list
    mach_rows = []
    for m in machines:
        used       = machine_hours.get(m, 0)
        remaining  = AVAILABLE_HOURS - used
        parts_run  = [r["Part"] for r in plan if r["Machine"] == m]
        parts_str  = ", ".join(parts_run) if parts_run else "— idle —"
        co_count   = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
 
        if used >= AVAILABLE_HOURS - 0.5:
            util_status = "FULL"
        elif used >= AVAILABLE_HOURS * 0.85:
            util_status = "GOOD"
        elif used >= AVAILABLE_HOURS * 0.5:
            util_status = "PARTIAL"
        else:
            util_status = "UNDERUSED"
 
        mach_rows.append({
            "Machine":           m,
            "Total_Available_Hrs": AVAILABLE_HOURS,
            "Used_Hours":        round(used, 2),
            "Unused_Hours":      round(remaining, 2),
            "Utilization_%":     round(used / AVAILABLE_HOURS * 100, 1),
            "Status":            util_status,
            "Parts_Planned":     len(parts_run),
            "Changeovers":       co_count,
            "Last_Part_Run":     machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": parts_str,
        })
 
    plan_df     = pd.DataFrame(plan)        if plan        else pd.DataFrame()
    def_df      = pd.DataFrame(deferred)    if deferred    else pd.DataFrame()
    not_df      = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df     = pd.DataFrame(mach_rows)
    inv_df      = pd.DataFrame(inv_rows)
 
    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
 
    return plan_df, def_df, not_df, mach_df, inv_df, machine_last_part, horizon_df
 
# =============================================================
# SECTION 16 — RUN BOTH MACHINE GROUPS
# =============================================================
 
hz_parts = data[data["Material"].isin(hz_matrix["Part"])]["Material"].unique()
vt_parts = data[data["Material"].isin(vt_matrix["Part"])]["Material"].unique()
 
hz_plan, hz_def, hz_not, hz_mach, hz_inv, hz_state, hz_horizon = \
    schedule(hz_parts, hz_compat, hz_machines, "HZ Machines")
 
vt_plan, vt_def, vt_not, vt_mach, vt_inv, vt_state, vt_horizon = \
    schedule(vt_parts, vt_compat, vt_machines, "VT Machines")
 
save_machine_state(hz_state, vt_state)
 
# =============================================================
# SECTION 17 — SAVE OUTPUT
# =============================================================
 
# =============================================================
# SECTION 17 — SAVE OUTPUT  (formatted Excel)
# =============================================================
 
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
 
# Colour palette for header rows
HEADER_COLORS = {
    "HZ_Plan":               "1F4E79",   # dark blue
    "VT_Plan":               "1F4E79",
    "HZ_Machine_Util":       "375623",   # dark green
    "VT_Machine_Util":       "375623",
    "HZ_Not_Planned":        "7B2C2C",   # dark red
    "VT_Not_Planned":        "7B2C2C",
    "HZ_Not_Required_Today":           "7F6000",   # dark amber
    "VT_Not_Required_Today":           "7F6000",
    "HZ_Inventory_Health":   "4A235A",   # dark purple
    "VT_Inventory_Health":   "4A235A",
    "HZ_Demand_Horizon":     "154360",
    "VT_Demand_Horizon":     "154360",
    "ALL_Plan_Combined":     "1F4E79",
}
 
STATUS_FILLS = {
    "FULL":      PatternFill("solid", fgColor="C6EFCE"),   # green
    "GOOD":      PatternFill("solid", fgColor="DDEBF7"),   # blue
    "PARTIAL":   PatternFill("solid", fgColor="FFEB9C"),   # yellow
    "UNDERUSED": PatternFill("solid", fgColor="FFC7CE"),   # red
    "OK":        PatternFill("solid", fgColor="C6EFCE"),
    "LOW":       PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":  PatternFill("solid", fgColor="FFC7CE"),
    "PRE-BUILD NEEDED": PatternFill("solid", fgColor="FFC7CE"),
    "WATCH — near capacity": PatternFill("solid", fgColor="FFEB9C"),
}
 
def style_sheet(ws, header_hex):
    """Apply header formatting and auto column widths to a worksheet."""
    header_fill = PatternFill("solid", fgColor=header_hex)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    center      = Alignment(horizontal="center", vertical="center", wrap_text=True)
 
    # Style header row
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center
 
    ws.row_dimensions[1].height = 32
 
    # Auto-fit column widths based on content
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                cell_len = len(str(cell.value)) if cell.value is not None else 0
                max_len  = max(max_len, cell_len)
            except Exception:
                pass
        # Cap between 10 and 50 characters wide
        ws.column_dimensions[col_letter].width = max(10, min(50, max_len + 3))
 
    # Colour status/flag cells
    status_col_names = {"Status", "D3_Flag", "Utilization_Status"}
    header_row = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(header_row, start=1):
        if col_name in status_col_names or "Status" in str(col_name):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value), None)
                    if fill:
                        cell.fill = fill
 
    # Freeze the header row
    ws.freeze_panes = "A2"
 
print(f"\nWriting → {output_path}")
 
all_plan = pd.concat([hz_plan, vt_plan], ignore_index=True)
 
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
 
    # ── Plans ────────────────────────────────────────────────
    hz_plan.to_excel(writer, sheet_name="HZ_Plan",           index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan",           index=False)
 
    # ── Machine utilization ──────────────────────────────────
    hz_mach.to_excel(writer, sheet_name="HZ_Machine_Util",   index=False)
    vt_mach.to_excel(writer, sheet_name="VT_Machine_Util",   index=False)
 
    # ── Not planned (capacity exhausted) ────────────────────
    hz_not.to_excel(writer,  sheet_name="HZ_Not_Planned",    index=False)
    vt_not.to_excel(writer,  sheet_name="VT_Not_Planned",    index=False)
 
    # ── Deferred (inventory sufficient) ─────────────────────
    hz_def.to_excel(writer,  sheet_name="HZ_Not_Required_Today",       index=False)
    vt_def.to_excel(writer,  sheet_name="VT_Not_Required_Today",       index=False)
 
    # ── Inventory health ─────────────────────────────────────
    hz_inv.to_excel(writer,  sheet_name="HZ_Inventory_Health", index=False)
    vt_inv.to_excel(writer,  sheet_name="VT_Inventory_Health", index=False)
 
    # ── Demand horizon ───────────────────────────────────────
    hz_horizon.to_excel(writer, sheet_name="HZ_Demand_Horizon", index=False)
    vt_horizon.to_excel(writer, sheet_name="VT_Demand_Horizon", index=False)
 
    # ── Combined plan ────────────────────────────────────────
    all_plan.to_excel(writer, sheet_name="ALL_Plan_Combined", index=False)
 
# Apply formatting after writing (openpyxl post-process)
wb = load_workbook(output_path)
for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames:
        style_sheet(wb[sheet_name], header_hex)
 
# Set sheet tab order — most important sheets first
tab_order = [
    "HZ_Plan", "VT_Plan",
    "HZ_Machine_Util", "VT_Machine_Util",
    "HZ_Not_Planned", "VT_Not_Planned",
    "HZ_Not_Required_Today", "VT_Not_Required_Today",
    "HZ_Inventory_Health", "VT_Inventory_Health",
    "HZ_Demand_Horizon", "VT_Demand_Horizon",
    "ALL_Plan_Combined",
]
for i, name in enumerate(tab_order):
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = (
            "1F4E79" if "Plan" in name else
            "375623" if "Machine" in name else
            "7B2C2C" if "Not_Planned" in name else
            "7F6000" if "Deferred" in name else
            "4A235A"
        )
 
wb.save(output_path)
print(f"  Formatting applied  ✓")
 
# =============================================================
# SECTION 18 — SUMMARY PRINT
# =============================================================
 
print(f"\n{'='*62}")
print(f"  Smart APS V4 Complete  —  {PLANNING_DATE}")
print(f"{'='*62}")
print(f"  HZ  planned={len(hz_plan):>4}  not_required_today={len(hz_def):>4}  not_planned={len(hz_not):>4}")
print(f"  VT  planned={len(vt_plan):>4}  not_required_today={len(vt_def):>4}  not_planned={len(vt_not):>4}")
print(f"\n  not_required_today = inventory already sufficient, no action needed")
print(f"  not_planned        = production was needed but no machine capacity available")
 
all_horizon = pd.concat([hz_horizon, vt_horizon], ignore_index=True)
pre_build   = all_horizon[all_horizon["D3_Pre_Build_Buffer"] > 0]
 
if not pre_build.empty:
    print(f"\n  D+3 (17th) pre-build buffers applied to {len(pre_build)} parts:")
    print(f"  {'Part':<30}  {'15th demand':>12}  {'17th tent':>10}  {'17th risk':>10}  {'buffer':>8}")
    print(f"  {'─'*30}  {'─'*12}  {'─'*10}  {'─'*10}  {'─'*8}")
    for _, r in pre_build.iterrows():
        print(f"  {r['Part']:<30}  {r['D1_Actual_15th']:>12.0f}  "
              f"{r['D3_Tentative_17th']:>10.0f}  "
              f"{r['D3_Risk_Adjusted']:>10.0f}  "
              f"{r['D3_Pre_Build_Buffer']:>8.0f}")
else:
    print(f"\n  No D+3 pre-build needed — 16th and 17th machine capacity is sufficient")
 
print(f"\n  Output  → {output_path}")
print(f"  State   → {MACHINE_STATE_FILE}")
print(f"\n  NOTE: Change the 3 lines in SECTION 1 each morning before running.")
print(f"        Tomorrow (15th): ACTUAL_DEMAND_COL    = '2026-03-13 Total Production Plan'")
print(f"                         TENTATIVE_DEMAND_COL = '2026-03-15 Total Production Plan'")
print(f"                         PLANNING_DATE        = date(2026, 3, 15)")


  Smart APS V4  —  Planning date: 2026-03-11
  Actual demand col   : 2026-03-12 Total Production Plan
  Tentative demand col: 2026-03-14 Total Production Plan
  16th March          : IGNORED today — will plan on 15th

Loading data...
Validating demand columns...
  Actual demand col        : '2026-03-12 Total Production Plan'  ✓
  Tentative demand col     : '2026-03-14 Total Production Plan'  ✓
  Machine state loaded  :
    BOY-10T-I M027            last ran → S21021-011A0X
    BOY-10T-II M028           last ran → S33047-005A0X
    BOY-10T-III M029          last ran → S01100-002A0X
    BOY-22T-I M030            last ran → 14SW410464-00002X2
    BOY-22T-II M031           last ran → 14SW410464-00003X2
    BOY-22T-III M032          last ran → 14SW410464-00004X2
    BOY-22T-IV M033           last ran → 14SW410568-00013X0
    BOY-22T-V M050            last ran → 14SW410464-00001X2
    BOY-22T-VI M051           last ran → 14SW410568-00008X0
    FANUC-50T-I M035          last ran → S33082-008

In [11]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, date

# =============================================================
# ██████╗  █████╗ ██████╗  █████╗ ███╗   ███╗███████╗
# ██╔══██╗██╔══██╗██╔══██╗██╔══██╗████╗ ████║██╔════╝
# ██████╔╝███████║██████╔╝███████║██╔████╔██║███████╗
# ██╔═══╝ ██╔══██║██╔══██╗██╔══██║██║╚██╔╝██║╚════██║
# ██║     ██║  ██║██║  ██║██║  ██║██║ ╚═╝ ██║███████║
# ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═╝╚═╝     ╚═╝╚══════╝
#
#  Smart APS V4  —  Forward-Looking Demand Horizon
#  Planning date  : 14th March 2026
#  Actual col     : "2026-03-12 Total Production Plan"  (supply on 15th)
#  Tentative col  : "2026-03-14 Total Production Plan"  (pre-build for 17th)
#  16th March     : ignored today, planned on 15th when data arrives
# =============================================================

# =============================================================
# SECTION 1 — COLUMN NAMES  (change these every day)
# -------------------------------------------------------------
# On 14th March your demand file has:
#   "2026-03-12 Total Production Plan" → actual for 15th  (D+1)
#   "2026-03-14 Total Production Plan" → tentative for 17th (D+3)
#
# Every morning update the two column name strings below
# to match whatever columns appear in that day's file.
# The planning date is only used for naming the output file.
# =============================================================

ACTUAL_DEMAND_COL    = "2026-03-12 Total Production Plan"   # D+1  ← change daily
TENTATIVE_DEMAND_COL = "2026-03-14 Total Production Plan"   # D+3  ← change daily
PLANNING_DATE        = date(2026, 3, 14)                    # today ← change daily

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS    = 22       # total machine hours per shift
MIN_RUN_HOURS      = 4        # minimum block per part on a machine
TARGET_DAYS_INV    = 3        # ideal inventory buffer (days)
MACHINE_STATE_FILE = "machine_state.json"

# Z-score for tentative demand buffer
# Since your tentative is nearly accurate → Z = 1.28 (90% coverage, modest buffer)
# If tentative were very volatile we'd use 1.65 or 2.0
Z_FOR_TENTATIVE    = 1.28

# Priority weights
W_COVERAGE   = 2.0
W_URGENCY    = 3.0
W_RISK       = 1.5
W_D3_BOOST   = 2.0   # how strongly D+3 pre-build need lifts priority

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path   = "C:/Users/Ex0164/Book1.xlsx"
daily_path  = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
output_path = f"Smart_APS_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — LOAD DATA
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V4  —  Planning date: {PLANNING_DATE}")
print(f"  Actual demand col   : {ACTUAL_DEMAND_COL}")
print(f"  Tentative demand col: {TENTATIVE_DEMAND_COL}")
print(f"  16th March          : IGNORED today — will plan on 15th")
print(f"{'='*62}\n")

print("Loading data...")
stats     = pd.read_excel(book_path,   sheet_name="Sheet2")
daily     = pd.read_excel(daily_path,  sheet_name="Sheet1")
hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

# =============================================================
# SECTION 5 — VALIDATE DEMAND COLUMNS
# -------------------------------------------------------------
# Columns are read directly by name — no date parsing needed.
# If a column is missing, a clear error is raised showing what
# columns are actually in the file so you know what to fix.
# =============================================================

def validate_column(df, col_name, label):
    """Check the column exists; if not, print all columns and raise."""
    if col_name in df.columns:
        print(f"  {label:25s}: '{col_name}'  ✓")
        return
    raise ValueError(
        f"\n  ERROR: Column not found — {col_name}\n"
        f"  Label : {label}\n"
        f"  Fix   : Update {label.split()[0].upper()}_DEMAND_COL in Section 1\n"
        f"  Columns available in file:\n"
        + "\n".join(f"    '{c}'" for c in df.columns)
    )

print("Validating demand columns...")
validate_column(daily, ACTUAL_DEMAND_COL,    "Actual demand col")
validate_column(daily, TENTATIVE_DEMAND_COL, "Tentative demand col")

# =============================================================
# SECTION 6 — MERGE & PRODUCTION RATE
# =============================================================

data = stats.merge(daily, left_on="Part", right_on="Material")

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]
data = data[data["Rate"].notna()].copy()

# =============================================================
# SECTION 7 — BUILD LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        k: (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
    }

inventory       = safe_dict(data, "Material", "Inventory on 24th")
rate            = safe_dict(data, "Material", "Rate")

# D+1: actual demand — must supply this (15th March)
demand_d1       = safe_dict(data, "Material", ACTUAL_DEMAND_COL)

# D+3: tentative demand — pre-build buffer for this (17th March)
demand_d3_tent  = safe_dict(data, "Material", TENTATIVE_DEMAND_COL)

# Historical stats (for smarter Z-score if available)
mean_demand = (safe_dict(data, "Material", "Mean_Demand")
               if "Mean_Demand" in data.columns else demand_d1.copy())
std_demand  = (safe_dict(data, "Material", "Std_Dev_Demand")
               if "Std_Dev_Demand" in data.columns
               else {k: 0.0 for k in demand_d1})

# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# =============================================================

# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# -------------------------------------------------------------
# This file is created automatically after the first run.
# On first run (file does not exist yet):
#   → No changeover penalty applied to anyone — clean slate
#   → After planning, the file is written with today's last parts
# From second run onwards:
#   → File is read, changeover penalties apply correctly
#   → Parts that ran last on a machine get zero changeover cost
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"  Machine state loaded  :")
        for m, p in state.items():
            print(f"    {m:25s} last ran → {p}")
        return state
    # First run — no state file exists yet
    print("  Machine state         : FIRST RUN — no previous state")
    print("                          No changeover penalties applied today.")
    print(f"                          State will be saved to '{MACHINE_STATE_FILE}'")
    print("                          after this run for use tomorrow.")
    return {}   # empty dict → machine_last_part[m] = None → no penalty

def save_machine_state(hz_state, vt_state):
    combined = {**hz_state, **vt_state}
    # Remove machines where last part is None (never ran)
    combined = {m: p for m, p in combined.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → {MACHINE_STATE_FILE}")
    print("  (Tomorrow's plan will use this to avoid unnecessary changeovers)")
    for m, p in combined.items():
        print(f"    {m:25s} last ran → {p}")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

hz_compat, hz_machines = build_compatibility(hz_matrix)
vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — DEMAND HORIZON ANALYSIS
# -------------------------------------------------------------
# For each part, calculate:
#
#   d1_shortage   = max(0, D+1 demand − current inventory)
#                 = what we MUST produce today for 15th supply
#
#   d3_buffer     = extra to pre-build today for 17th demand
#                 = max(0, risk_adjusted_D3 − inv_at_d3_start
#                           − machine_capacity_on_16th
#                           − machine_capacity_on_17th)
#
#   today_target  = d1_shortage + d3_buffer
#
# Why do we subtract capacity on 16th AND 17th?
#   Because even without a demand file for 16th, we can still
#   PRODUCE on 16th. So the pre-build needed today is only what
#   BOTH 16th and 17th combined cannot cover.
#
# Note: Since tentative is nearly accurate, Z = 1.28 (modest buffer)
# =============================================================

def compute_demand_horizon(parts, compatibility, machines):

    def max_daily_capacity(part):
        """Max units producible in one full day across all compatible machines."""
        compat_machines = compatibility.get(part, [])
        if not compat_machines:
            return 0
        r = rate.get(part, 1)
        # Each machine can run AVAILABLE_HOURS independently
        return len(compat_machines) * AVAILABLE_HOURS * r

    rows = []
    for p in parts:
        inv      = inventory.get(p, 0)
        d1       = demand_d1.get(p, 0)
        d3_t     = demand_d3_tent.get(p, 0)
        mean     = mean_demand.get(p, max(d1, 1))
        std      = std_demand.get(p, 0)
        r        = rate.get(p, 1)

        # ── D+1 (15th) shortage ─────────────────────────────
        d1_shortage    = max(0.0, d1 - inv)
        inv_after_d1   = max(0.0, inv - d1)   # inventory left after supplying 15th

        # ── D+3 (17th) risk-adjusted target ─────────────────
        # Since tentative is nearly accurate, small Z = 1.28
        d3_risk_adj    = d3_t + Z_FOR_TENTATIVE * std

        # ── Capacity available on 16th and 17th ─────────────
        # 16th: no demand file, machines run freely → full capacity available
        # 17th: machines run freely → full capacity available
        # Together they can cover: 2 × daily_capacity
        cap_16th       = max_daily_capacity(p)
        cap_17th       = max_daily_capacity(p)

        # Inventory at start of 17th =
        #   inv_after_d1 (left after 15th supply)
        #   + whatever we produce on 16th (up to cap_16th, but only if needed)
        # We assume 16th production goes toward 17th demand as well
        # So total coverage for 17th = inv_after_d1 + cap_16th + cap_17th

        total_coverage_for_d3 = inv_after_d1 + cap_16th + cap_17th

        # Pre-build needed today = gap that even 16th+17th machines can't cover
        d3_buffer = max(0.0, d3_risk_adj - total_coverage_for_d3)

        # ── Combined today's target ──────────────────────────
        today_target = d1_shortage + d3_buffer

        # ── D+3 status flag ─────────────────────────────────
        if d3_buffer > 0:
            d3_flag = "PRE-BUILD NEEDED"
        elif d3_risk_adj > (cap_16th + cap_17th) * 0.8:
            d3_flag = "WATCH — near capacity"
        else:
            d3_flag = "OK"

        rows.append({
            "Part":                   p,
            "Inventory_Now":          round(inv, 0),

            # D+1 (15th) columns
            "D1_Actual_15th":         round(d1, 0),
            "D1_Shortage":            round(d1_shortage, 0),
            "Inv_After_15th_Supply":  round(inv_after_d1, 0),

            # D+3 (17th) columns
            "D3_Tentative_17th":      round(d3_t, 0),
            "D3_Risk_Adjusted":       round(d3_risk_adj, 0),
            "Cap_16th_Available":     round(cap_16th, 0),
            "Cap_17th_Available":     round(cap_17th, 0),
            "D3_Pre_Build_Buffer":    round(d3_buffer, 0),
            "D3_Flag":                d3_flag,

            # Combined
            "Today_Total_Target_Qty": round(today_target, 0),
            "Today_Total_Target_Hrs": round(today_target / r if r > 0 else 0, 2),
        })

    return pd.DataFrame(rows)

# =============================================================
# SECTION 11 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv  = inventory.get(p, 0)
        d1   = demand_d1.get(p, 0)
        mean = mean_demand.get(p, d1)
        # Parts with zero demand are not critical regardless of inventory
        if d1 == 0 and mean == 0:
            continue
        daily = max(mean, d1, 1)
        coverage.append(inv / daily)

    if not coverage:
        # All parts have zero demand — best case
        return 3, 1.28, "SCENARIO 3 — All parts healthy (zero demand today)"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, 2.00, "SCENARIO 0 — ALL parts critical (inv < 1 day demand)"
    elif critical > 0:
        return 1, 1.65, f"SCENARIO 1 — {critical}/{n} parts critical (inv < 1 day)"
    elif low > 0:
        return 2, 1.28, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, 1.28, f"SCENARIO 3 — All parts healthy (≥{TARGET_DAYS_INV} days inv)"

# =============================================================
# SECTION 12 — PRIORITY SCORING  (D+3-aware)
# =============================================================

def compute_priority(parts, horizon_df):
    horizon = horizon_df.set_index("Part")
    rows    = []

    for p in parts:
        inv  = inventory.get(p, 0)
        d1   = demand_d1.get(p, 0)
        mean = mean_demand.get(p, max(d1, 1))
        std  = std_demand.get(p, 0)

        days_cov = inv / mean if mean > 0 else 999

        # Urgency based on D+1 coverage
        if days_cov < 1:
            urgency = 1.0
        elif days_cov < 2:
            urgency = 0.5
        else:
            urgency = 0.0

        cv = std / mean if mean > 0 else 0

        # D+3 boost: if pre-build buffer > 0, add extra urgency
        d3_buf  = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
        d3_boost = min(1.0, d3_buf / max(mean, 1))

        score = (W_COVERAGE * (1.0 / (days_cov + 0.01))
               + W_URGENCY  * urgency
               + W_RISK     * cv
               + W_D3_BOOST * d3_boost)

        today_target = (horizon.loc[p, "Today_Total_Target_Qty"]
                        if p in horizon.index else d1)

        rows.append({
            "Part":           p,
            "Days_Coverage":  round(days_cov, 2),
            "Urgency_D1":     urgency,
            "CV":             round(cv, 3),
            "D3_Boost":       round(d3_boost, 3),
            "Score":          round(score, 4),
            "D1_Demand_15th": round(d1, 0),
            "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            "D3_Pre_Build":   round(d3_buf, 0),
            "Today_Target":   round(today_target, 0),
        })

    return (pd.DataFrame(rows)
              .sort_values("Score", ascending=False)
              .reset_index(drop=True))

# =============================================================
# SECTION 13 — UBER-STYLE MACHINE RANKER
# =============================================================

def rank_machines(part, machines, compatibility, machine_hours, machine_last_part):
    """
    Returns machines sorted by assignment cost (lowest = best choice).
    Cost = changeover_penalty + load_factor
    Same part ran last on machine → 0 changeover penalty (zero cost bonus)
    """
    ranked = []
    for m in machines:
        if m not in compatibility.get(part, []):
            continue
        used = machine_hours.get(m, 0)
        free = AVAILABLE_HOURS - used
        if free < MIN_RUN_HOURS:
            continue
        last               = machine_last_part.get(m)
        changeover_penalty = 0.0 if last == part else 0.5
        cost               = changeover_penalty + (used / AVAILABLE_HOURS)
        ranked.append((m, cost, free, changeover_penalty))

    ranked.sort(key=lambda x: x[1])
    return ranked

# =============================================================
# SECTION 14 — 22-HOUR UTILIZATION FILLER
# =============================================================

def fill_remaining_hours(plan, machine_hours, machine_last_part,
                         parts, compatibility, machines,
                         current_inventory, horizon_df, scenario):
    """
    After primary assignments, fill every remaining minute of machine time.

    The MIN_RUN_HOURS (4h) check applies only to PRIMARY scheduling
    (first-time assignment of a new part to a machine).
    The filler has NO minimum — it uses whatever hours remain, even 1h or 2h,
    by extending an already-running part (zero changeover, any duration is fine).

    This fixes the 18-22h gap: a machine at 18.5h with 3.5h left would
    previously be skipped (3.5 < 4h minimum). Now it gets extended.
    """
    filler  = []
    horizon = horizon_df.set_index("Part")

    for m in machines:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
        if remaining <= 0:
            continue   # truly full — nothing to add

        on_machine = [row["Part"] for row in plan if row["Machine"] == m]

        # ── Option A: extend already-planned part ────────────
        # No minimum time check here — any remaining hours are useful.
        # Prefer the part with the largest gap to its 3-day inventory target.
        extended = False
        best_part, best_gap = None, -1
        for p in on_machine:
            inv_now    = current_inventory.get(p, 0)
            target_inv = mean_demand.get(p, demand_d1.get(p, 0)) * TARGET_DAYS_INV
            d3_buf     = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
            gap        = max(0, target_inv - inv_now) + d3_buf
            if gap > best_gap:
                best_gap, best_part = gap, p

        if best_part and best_gap > 0:
            p         = best_part
            r_val     = rate.get(p, 1)
            extra_qty = round(remaining * r_val, 0)
            machine_hours[m]    += remaining
            current_inventory[p] = current_inventory.get(p, 0) + extra_qty
            filler.append({
                "Part":           p,
                "Machine":        m,
                "Run_Hours":      round(remaining, 2),
                "Production_Qty": extra_qty,
                "Type":           ("Extend — D3 buffer"
                                   if (p in horizon.index and horizon.loc[p, "D3_Pre_Build_Buffer"] > 0)
                                   else "Extend — inv build"),
                "Changeover":     "No",
                "D1_Demand_15th": round(demand_d1.get(p, 0), 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            })
            extended = True

        if extended:
            continue

        # ── Option B: new part — only if remaining >= MIN_RUN_HOURS
        # For a BRAND NEW part on this machine we still require 4h minimum
        # because a changeover + very short run is wasteful.
        if remaining < MIN_RUN_HOURS:
            continue   # not enough time for a new part — leave as-is

        candidates = []
        for p in parts:
            if p in on_machine:
                continue
            if m not in compatibility.get(p, []):
                continue
            inv_now = current_inventory.get(p, 0)
            d3_buf  = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
            tgt_inv = mean_demand.get(p, demand_d1.get(p, 0)) * TARGET_DAYS_INV
            if scenario == 3 and inv_now >= tgt_inv and d3_buf == 0:
                continue
            candidates.append((p, d3_buf))

        candidates.sort(key=lambda x: -x[1])

        for p, d3_buf in candidates:
            last            = machine_last_part.get(m)
            changeover_flag = "No" if last == p else "Yes"
            r_val           = rate.get(p, 1)
            run_h           = remaining
            qty             = round(run_h * r_val, 0)

            machine_hours[m]    += run_h
            current_inventory[p] = current_inventory.get(p, 0) + qty
            machine_last_part[m] = p

            filler.append({
                "Part":           p,
                "Machine":        m,
                "Run_Hours":      round(run_h, 2),
                "Production_Qty": qty,
                "Type":           "New — D3 pre-build" if d3_buf > 0 else "New — inv fill",
                "Changeover":     changeover_flag,
                "D1_Demand_15th": round(demand_d1.get(p, 0), 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            })
            break

    return filler

# =============================================================
# SECTION 15 — MAIN SCHEDULER
# =============================================================

def schedule(parts, compatibility, machines, label=""):

    print(f"\n{'─'*62}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(machines)} machines")
    print(f"{'─'*62}")

    # Scenario
    scenario, _, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    # Demand horizon
    horizon_df = compute_demand_horizon(parts, compatibility, machines)

    pre_build_parts = horizon_df[horizon_df["D3_Pre_Build_Buffer"] > 0]
    if not pre_build_parts.empty:
        print(f"\n  D+3 pre-build required ({len(pre_build_parts)} parts):")
        for _, r in pre_build_parts.iterrows():
            print(f"    {r['Part']:30s}  "
                  f"15th={r['D1_Actual_15th']:.0f}  "
                  f"17th_tent={r['D3_Tentative_17th']:.0f}  "
                  f"17th_risk={r['D3_Risk_Adjusted']:.0f}  "
                  f"pre_build={r['D3_Pre_Build_Buffer']:.0f}")
    else:
        print("  No D+3 pre-build needed — 16th+17th capacity covers 17th demand")

    # State
    machine_hours     = {m: 0.0 for m in machines}
    machine_last_part = {m: machine_state.get(m) for m in machines}
    current_inventory = inventory.copy()

    plan, deferred, not_planned = [], [], []

    # Priority sort
    priority_df = compute_priority(parts, horizon_df)
    horizon_idx = horizon_df.set_index("Part")

    print(f"\n  Priority order:")
    for _, r in priority_df.iterrows():
        print(f"    {r['Part']:30s}  "
              f"score={r['Score']:.3f}  "
              f"cov={r['Days_Coverage']}d  "
              f"D1_short={max(0, r['D1_Demand_15th'] - inventory.get(r['Part'], 0)):.0f}  "
              f"D3_prebuild={r['D3_Pre_Build']:.0f}")

    # ── Main assignment loop ────────────────────────────────
    print(f"\n  Assignments:")
    for _, row in priority_df.iterrows():
        part     = row["Part"]
        inv_now  = current_inventory.get(part, 0)
        d1       = demand_d1.get(part, 0)
        d3_buf   = row["D3_Pre_Build"]
        r_val    = rate.get(part, 1)

        inv_covers_d1  = inv_now >= d1
        compatible_mch = compatibility.get(part, [])
        days_cov_now   = inv_now / max(mean_demand.get(part, max(d1, 1)), 1)

        # ── Decision 1: zero demand — never needs planning ─────
        # Regardless of scenario, if demand is 0 and no D+3 buffer
        # is needed, this part should not be planned today.
        if d1 == 0 and d3_buf == 0:
            deferred.append({
                "Part":              part,
                "Inventory_Now":     round(inv_now, 0),
                "D1_Demand_15th":    0,
                "Inventory_Surplus": round(inv_now, 0),
                "Days_Coverage":     round(days_cov_now, 2),
                "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                "Reason":            "Demand is zero — no production required today",
                "Next_Action":       "Will be re-evaluated when demand appears",
            })
            print(f"    - {part:30s}  NOT REQUIRED — demand = 0")
            continue

        # ── Decision 2: inventory sufficient, scenario not critical
        # In scenario 0/1 (critical), we MUST try to produce everything
        # even if inventory partially covers demand — machines must run.
        # In scenario 2/3 (healthy/moderate), if inventory covers D+1
        # and no D+3 buffer needed → safe to skip today.
        if inv_covers_d1 and d3_buf == 0 and scenario >= 2:
            deferred.append({
                "Part":              part,
                "Inventory_Now":     round(inv_now, 0),
                "D1_Demand_15th":    round(d1, 0),
                "Inventory_Surplus": round(inv_now - d1, 0),
                "Days_Coverage":     round(days_cov_now, 2),
                "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                "Reason":            "Inventory sufficient — covers D+1 demand with surplus",
                "Next_Action":       "Will be re-evaluated tomorrow",
            })
            print(f"    - {part:30s}  NOT REQUIRED TODAY"
                  f"  inv={inv_now:.0f}  d1={d1:.0f}  surplus={inv_now-d1:.0f}")
            continue

        # ── Decision 3: scenario 0/1 but inventory already covers D+1
        # AND no D+3 buffer needed — still skip, log separately
        if inv_covers_d1 and d3_buf == 0 and scenario < 2:
            deferred.append({
                "Part":              part,
                "Inventory_Now":     round(inv_now, 0),
                "D1_Demand_15th":    round(d1, 0),
                "Inventory_Surplus": round(inv_now - d1, 0),
                "Days_Coverage":     round(days_cov_now, 2),
                "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                "Reason":            "Inventory sufficient even in critical scenario — skipped to free capacity for critical parts",
                "Next_Action":       "Monitor — will be produced once critical parts are covered",
            })
            print(f"    - {part:30s}  NOT REQUIRED TODAY (critical scenario but inv covers D+1)")
            continue

        # ── Decision 4: part needs production — find a machine ─
        today_qty  = row["Today_Target"]
        # In scenario 0 (all critical): target = exactly D+1 shortage, no extras
        # In other scenarios: target includes D+3 buffer
        if scenario == 0:
            today_qty = max(0, d1 - inv_now)   # only what's needed to survive today
        target_hrs = max(MIN_RUN_HOURS, today_qty / r_val if r_val > 0 else MIN_RUN_HOURS)

        # Check if part even has compatible machines defined
        if not compatible_mch:
            not_planned.append({
                "Part":                part,
                "D1_Demand_15th":      round(d1, 0),
                "Inventory_Now":       round(inv_now, 0),
                "Shortage":            round(max(0, d1 - inv_now), 0),
                "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                "Compatible_Machines": "NONE DEFINED",
                "Reason":              "Part has no compatible machines in the matrix",
                "Action_Needed":       "Add this part to the compatibility matrix",
            })
            print(f"    ✗ {part:30s}  NOT PLANNED — not in compatibility matrix")
            continue

        # Uber-style machine ranking
        ranked   = rank_machines(part, machines, compatibility,
                                 machine_hours, machine_last_part)

        # All compatible machines exist but none have free hours
        if not ranked:
            # Determine precise reason
            all_full    = all(machine_hours.get(m, 0) >= AVAILABLE_HOURS - MIN_RUN_HOURS
                              for m in compatible_mch if m in machine_hours)
            not_in_group = [m for m in compatible_mch if m not in machine_hours]

            if not_in_group:
                reason  = f"Compatible machines {not_in_group} not in this machine group"
                action  = "Check compatibility matrix — machine may belong to wrong group (HZ/VT)"
            elif all_full:
                reason  = f"All compatible machines fully utilised ({', '.join(compatible_mch)})"
                action  = "Increase machine hours, reduce qty of lower-priority parts, or plan tomorrow"
            else:
                reason  = "Compatible machines exist but remaining hours below MIN_RUN_HOURS (4h)"
                action  = f"Reduce MIN_RUN_HOURS or free up capacity on {', '.join(compatible_mch)}"

            not_planned.append({
                "Part":                part,
                "D1_Demand_15th":      round(d1, 0),
                "Inventory_Now":       round(inv_now, 0),
                "Shortage":            round(max(0, d1 - inv_now), 0),
                "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                "Compatible_Machines": ", ".join(compatible_mch),
                "Reason":              reason,
                "Action_Needed":       action,
            })
            print(f"    ✗ {part:30s}  NOT PLANNED — {reason}")
            continue

        assigned = False

        for m, cost, free_h, co_penalty in ranked:
            run_h = min(target_hrs, free_h)
            run_h = max(run_h, MIN_RUN_HOURS)
            run_h = min(run_h, free_h)
            qty   = round(run_h * r_val, 0)

            machine_hours[m]        += run_h
            current_inventory[part]  = current_inventory.get(part, 0) + qty
            machine_last_part[m]     = part

            plan.append({
                "Part":           part,
                "Machine":        m,
                "Run_Hours":      round(run_h, 2),
                "Production_Qty": qty,
                "D1_Demand_15th": round(d1, 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(part, 0), 0),
                "D3_Pre_Build":   round(d3_buf, 0),
                "Changeover":     "No" if co_penalty == 0 else "Yes",
                "Type":           "Primary",
                "Priority_Score": row["Score"],
            })

            co_str = "No " if co_penalty == 0 else "Yes"
            print(f"    ✓ {part:30s} → {m:15s}  "
                  f"{run_h:.2f}h  qty={qty:>8.0f}  "
                  f"CO={co_str}  "
                  f"[D1={d1:.0f} | D3buf={d3_buf:.0f}]")
            assigned = True
            break

        if not assigned:
            # Try splitting across multiple compatible machines
            remaining_target = target_hrs
            split_done       = False

            for m, cost, free_h, co_penalty in ranked:
                if free_h < MIN_RUN_HOURS or remaining_target <= 0:
                    break
                run_h = min(remaining_target, free_h)
                run_h = max(run_h, MIN_RUN_HOURS)
                qty   = round(run_h * r_val, 0)

                machine_hours[m]        += run_h
                current_inventory[part]  = current_inventory.get(part, 0) + qty
                machine_last_part[m]     = part
                remaining_target        -= run_h

                plan.append({
                    "Part":           part,
                    "Machine":        m,
                    "Run_Hours":      round(run_h, 2),
                    "Production_Qty": qty,
                    "D1_Demand_15th": round(d1, 0),
                    "D3_Tent_17th":   round(demand_d3_tent.get(part, 0), 0),
                    "D3_Pre_Build":   round(d3_buf, 0),
                    "Changeover":     "No" if co_penalty == 0 else "Yes",
                    "Type":           "Split",
                    "Priority_Score": row["Score"],
                })
                print(f"    ↔ {part:30s} → {m:15s}  "
                      f"{run_h:.2f}h  [SPLIT]")
                split_done = True

            if not split_done:
                # Ranked was non-empty but every machine was too full for even MIN_RUN_HOURS
                machines_status = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                                   for m in compatible_mch if m in machine_hours}
                reason = ("All compatible machines have less than MIN_RUN_HOURS "
                          f"({MIN_RUN_HOURS}h) remaining. "
                          f"Free hours: "
                          + ", ".join(f"{m}={h}h" for m, h in machines_status.items()))

                not_planned.append({
                    "Part":                part,
                    "D1_Demand_15th":      round(d1, 0),
                    "Inventory_Now":       round(inv_now, 0),
                    "Shortage":            round(max(0, d1 - inv_now), 0),
                    "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              reason,
                    "Action_Needed":       "Inventory will need to cover demand — check Inventory_Health sheet",
                })
                print(f"    ✗ {part:30s}  NOT PLANNED — capacity too fragmented")

    # 22-hour filler
    print(f"\n  22-hour filler:")
    filler_rows = fill_remaining_hours(
        plan, machine_hours, machine_last_part,
        list(parts), compatibility, machines,
        current_inventory, horizon_df, scenario
    )
    if filler_rows:
        for fr in filler_rows:
            plan.append(fr)
            print(f"    + {fr['Part']:30s} → {fr['Machine']:15s}  "
                  f"{fr['Run_Hours']}h  [{fr['Type']}]  CO={fr['Changeover']}")
    else:
        print("    All machines fully utilised — no filler needed")

    # ── Inventory health after today's plan ─────────────────
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        d1       = demand_d1.get(p, 0)
        d3_t     = demand_d3_tent.get(p, 0)
        produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)

        # final_inventory = inventory + produced − demand_supplied
        inv_after = inv_b + produced - d1
        mean      = mean_demand.get(p, max(d1, 1))
        days_cov  = inv_after / mean if mean > 0 else 0

        # How much of 17th tentative is covered by this inventory?
        d3_covered  = min(inv_after, d3_t)
        d3_still_gap = max(0, d3_t - inv_after)

        inv_rows.append({
            "Part":                   p,
            "Inv_Before":             round(inv_b, 0),
            "Produced_14th":          round(produced, 0),
            "D1_Supplied_15th":       round(d1, 0),
            "Inv_After_15th_Supply":  round(inv_after, 0),   # = inv + produced - d1
            "Days_Coverage":          round(days_cov, 2),
            "D3_Tentative_17th":      round(d3_t, 0),
            "D3_Covered_By_Inv":      round(d3_covered, 0),
            "D3_Still_Needs_16th_17th": round(d3_still_gap, 0),
            "Status":                 ("OK"       if days_cov >= 1
                                       else "CRITICAL" if inv_after < 0
                                       else "LOW"),
        })

    # Machine utilization — rich columns including parts list
    mach_rows = []
    for m in machines:
        used       = machine_hours.get(m, 0)
        remaining  = AVAILABLE_HOURS - used
        parts_run  = [r["Part"] for r in plan if r["Machine"] == m]
        parts_str  = ", ".join(parts_run) if parts_run else "— idle —"
        co_count   = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")

        if used >= AVAILABLE_HOURS - 0.5:
            util_status = "FULL"
        elif used >= AVAILABLE_HOURS * 0.85:
            util_status = "GOOD"
        elif used >= AVAILABLE_HOURS * 0.5:
            util_status = "PARTIAL"
        else:
            util_status = "UNDERUSED"

        mach_rows.append({
            "Machine":           m,
            "Total_Available_Hrs": AVAILABLE_HOURS,
            "Used_Hours":        round(used, 2),
            "Unused_Hours":      round(remaining, 2),
            "Utilization_%":     round(used / AVAILABLE_HOURS * 100, 1),
            "Status":            util_status,
            "Parts_Planned":     len(parts_run),
            "Changeovers":       co_count,
            "Last_Part_Run":     machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": parts_str,
        })

    plan_df     = pd.DataFrame(plan)        if plan        else pd.DataFrame()
    def_df      = pd.DataFrame(deferred)    if deferred    else pd.DataFrame()
    not_df      = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df     = pd.DataFrame(mach_rows)
    inv_df      = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)

    return plan_df, def_df, not_df, mach_df, inv_df, machine_last_part, horizon_df

# =============================================================
# SECTION 16 — RUN BOTH MACHINE GROUPS
# =============================================================

hz_parts = data[data["Material"].isin(hz_matrix["Part"])]["Material"].unique()
vt_parts = data[data["Material"].isin(vt_matrix["Part"])]["Material"].unique()

hz_plan, hz_def, hz_not, hz_mach, hz_inv, hz_state, hz_horizon = \
    schedule(hz_parts, hz_compat, hz_machines, "HZ Machines")

vt_plan, vt_def, vt_not, vt_mach, vt_inv, vt_state, vt_horizon = \
    schedule(vt_parts, vt_compat, vt_machines, "VT Machines")

save_machine_state(hz_state, vt_state)

# =============================================================
# SECTION 17 — SAVE OUTPUT
# =============================================================

# =============================================================
# SECTION 17 — SAVE OUTPUT  (formatted Excel)
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Colour palette for header rows
HEADER_COLORS = {
    "HZ_Plan":               "1F4E79",   # dark blue
    "VT_Plan":               "1F4E79",
    "HZ_Machine_Util":       "375623",   # dark green
    "VT_Machine_Util":       "375623",
    "HZ_Not_Planned":        "7B2C2C",   # dark red
    "VT_Not_Planned":        "7B2C2C",
    "HZ_Not_Required_Today":           "7F6000",   # dark amber
    "VT_Not_Required_Today":           "7F6000",
    "HZ_Inventory_Health":   "4A235A",   # dark purple
    "VT_Inventory_Health":   "4A235A",
    "HZ_Demand_Horizon":     "154360",
    "VT_Demand_Horizon":     "154360",
    "ALL_Plan_Combined":     "1F4E79",
}

STATUS_FILLS = {
    "FULL":      PatternFill("solid", fgColor="C6EFCE"),   # green
    "GOOD":      PatternFill("solid", fgColor="DDEBF7"),   # blue
    "PARTIAL":   PatternFill("solid", fgColor="FFEB9C"),   # yellow
    "UNDERUSED": PatternFill("solid", fgColor="FFC7CE"),   # red
    "OK":        PatternFill("solid", fgColor="C6EFCE"),
    "LOW":       PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":  PatternFill("solid", fgColor="FFC7CE"),
    "PRE-BUILD NEEDED": PatternFill("solid", fgColor="FFC7CE"),
    "WATCH — near capacity": PatternFill("solid", fgColor="FFEB9C"),
}

def style_sheet(ws, header_hex):
    """Apply header formatting and auto column widths to a worksheet."""
    header_fill = PatternFill("solid", fgColor=header_hex)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    center      = Alignment(horizontal="center", vertical="center", wrap_text=True)

    # Style header row
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center

    ws.row_dimensions[1].height = 32

    # Auto-fit column widths based on content
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                cell_len = len(str(cell.value)) if cell.value is not None else 0
                max_len  = max(max_len, cell_len)
            except Exception:
                pass
        # Cap between 10 and 50 characters wide
        ws.column_dimensions[col_letter].width = max(10, min(50, max_len + 3))

    # Colour status/flag cells
    status_col_names = {"Status", "D3_Flag", "Utilization_Status"}
    header_row = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(header_row, start=1):
        if col_name in status_col_names or "Status" in str(col_name):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value), None)
                    if fill:
                        cell.fill = fill

    # Freeze the header row
    ws.freeze_panes = "A2"

print(f"\nWriting → {output_path}")

all_plan = pd.concat([hz_plan, vt_plan], ignore_index=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # ── Plans ────────────────────────────────────────────────
    hz_plan.to_excel(writer, sheet_name="HZ_Plan",           index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan",           index=False)

    # ── Machine utilization ──────────────────────────────────
    hz_mach.to_excel(writer, sheet_name="HZ_Machine_Util",   index=False)
    vt_mach.to_excel(writer, sheet_name="VT_Machine_Util",   index=False)

    # ── Not planned (capacity exhausted) ────────────────────
    hz_not.to_excel(writer,  sheet_name="HZ_Not_Planned",    index=False)
    vt_not.to_excel(writer,  sheet_name="VT_Not_Planned",    index=False)

    # ── Deferred (inventory sufficient) ─────────────────────
    hz_def.to_excel(writer,  sheet_name="HZ_Not_Required_Today",       index=False)
    vt_def.to_excel(writer,  sheet_name="VT_Not_Required_Today",       index=False)

    # ── Inventory health ─────────────────────────────────────
    hz_inv.to_excel(writer,  sheet_name="HZ_Inventory_Health", index=False)
    vt_inv.to_excel(writer,  sheet_name="VT_Inventory_Health", index=False)

    # ── Demand horizon ───────────────────────────────────────
    hz_horizon.to_excel(writer, sheet_name="HZ_Demand_Horizon", index=False)
    vt_horizon.to_excel(writer, sheet_name="VT_Demand_Horizon", index=False)

    # ── Combined plan ────────────────────────────────────────
    all_plan.to_excel(writer, sheet_name="ALL_Plan_Combined", index=False)

# Apply formatting after writing (openpyxl post-process)
wb = load_workbook(output_path)
for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames:
        style_sheet(wb[sheet_name], header_hex)

# Set sheet tab order — most important sheets first
tab_order = [
    "HZ_Plan", "VT_Plan",
    "HZ_Machine_Util", "VT_Machine_Util",
    "HZ_Not_Planned", "VT_Not_Planned",
    "HZ_Not_Required_Today", "VT_Not_Required_Today",
    "HZ_Inventory_Health", "VT_Inventory_Health",
    "HZ_Demand_Horizon", "VT_Demand_Horizon",
    "ALL_Plan_Combined",
]
for i, name in enumerate(tab_order):
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = (
            "1F4E79" if "Plan" in name else
            "375623" if "Machine" in name else
            "7B2C2C" if "Not_Planned" in name else
            "7F6000" if "Deferred" in name else
            "4A235A"
        )

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 18 — SUMMARY PRINT
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V4 Complete  —  {PLANNING_DATE}")
print(f"{'='*62}")
print(f"  HZ  planned={len(hz_plan):>4}  not_required_today={len(hz_def):>4}  not_planned={len(hz_not):>4}")
print(f"  VT  planned={len(vt_plan):>4}  not_required_today={len(vt_def):>4}  not_planned={len(vt_not):>4}")
print(f"\n  not_required_today = inventory already sufficient, no action needed")
print(f"  not_planned        = production was needed but no machine capacity available")

all_horizon = pd.concat([hz_horizon, vt_horizon], ignore_index=True)
pre_build   = all_horizon[all_horizon["D3_Pre_Build_Buffer"] > 0]

if not pre_build.empty:
    print(f"\n  D+3 (17th) pre-build buffers applied to {len(pre_build)} parts:")
    print(f"  {'Part':<30}  {'15th demand':>12}  {'17th tent':>10}  {'17th risk':>10}  {'buffer':>8}")
    print(f"  {'─'*30}  {'─'*12}  {'─'*10}  {'─'*10}  {'─'*8}")
    for _, r in pre_build.iterrows():
        print(f"  {r['Part']:<30}  {r['D1_Actual_15th']:>12.0f}  "
              f"{r['D3_Tentative_17th']:>10.0f}  "
              f"{r['D3_Risk_Adjusted']:>10.0f}  "
              f"{r['D3_Pre_Build_Buffer']:>8.0f}")
else:
    print(f"\n  No D+3 pre-build needed — 16th and 17th machine capacity is sufficient")

print(f"\n  Output  → {output_path}")
print(f"  State   → {MACHINE_STATE_FILE}")
print(f"\n  NOTE: Change the 3 lines in SECTION 1 each morning before running.")
print(f"        Tomorrow (15th): ACTUAL_DEMAND_COL    = '2026-03-13 Total Production Plan'")
print(f"                         TENTATIVE_DEMAND_COL = '2026-03-15 Total Production Plan'")
print(f"                         PLANNING_DATE        = date(2026, 3, 15)")


  Smart APS V4  —  Planning date: 2026-03-14
  Actual demand col   : 2026-03-12 Total Production Plan
  Tentative demand col: 2026-03-14 Total Production Plan
  16th March          : IGNORED today — will plan on 15th

Loading data...
Validating demand columns...
  Actual demand col        : '2026-03-12 Total Production Plan'  ✓
  Tentative demand col     : '2026-03-14 Total Production Plan'  ✓
  Machine state loaded  :
    BOY-10T-I M027            last ran → S32047-011A0X
    BOY-10T-II M028           last ran → S33047-005A0X
    BOY-10T-III M029          last ran → S01100-002A0X
    BOY-22T-I M030            last ran → 14SW410464-00002X2
    BOY-22T-II M031           last ran → 14SW410464-00003X2
    BOY-22T-III M032          last ran → 14SW410464-00004X2
    BOY-22T-IV M033           last ran → 14SW410568-00013X0
    BOY-22T-V M050            last ran → 14SW410464-00001X2
    BOY-22T-VI M051           last ran → 14SW410568-00008X0
    FANUC-50T-I M035          last ran → S33082-008

In [9]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, date

# =============================================================
# ██████╗  █████╗ ██████╗  █████╗ ███╗   ███╗███████╗
# ██╔══██╗██╔══██╗██╔══██╗██╔══██╗████╗ ████║██╔════╝
# ██████╔╝███████║██████╔╝███████║██╔████╔██║███████╗
# ██╔═══╝ ██╔══██║██╔══██╗██╔══██║██║╚██╔╝██║╚════██║
# ██║     ██║  ██║██║  ██║██║  ██║██║ ╚═╝ ██║███████║
# ╚═╝     ╚═╝  ╚═╝╚═╝  ╚═╝╚═╝  ╚═╝╚═╝     ╚═╝╚══════╝
#
#  Smart APS V4  —  Forward-Looking Demand Horizon
#  Planning date  : 14th March 2026
#  Actual col     : "2026-03-12 Total Production Plan"  (supply on 15th)
#  Tentative col  : "2026-03-14 Total Production Plan"  (pre-build for 17th)
#  16th March     : ignored today, planned on 15th when data arrives
# =============================================================

# =============================================================
# SECTION 1 — COLUMN NAMES  (change these every day)
# -------------------------------------------------------------
# On 14th March your demand file has:
#   "2026-03-12 Total Production Plan" → actual for 15th  (D+1)
#   "2026-03-14 Total Production Plan" → tentative for 17th (D+3)
#
# Every morning update the two column name strings below
# to match whatever columns appear in that day's file.
# The planning date is only used for naming the output file.
# =============================================================

ACTUAL_DEMAND_COL    = "2026-03-12 Total Production Plan"   # D+1  ← change daily
TENTATIVE_DEMAND_COL = "2026-03-14 Total Production Plan"   # D+3  ← change daily
PLANNING_DATE        = date(2026, 3, 14)                    # today ← change daily

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS    = 22       # total machine hours per shift
MIN_RUN_HOURS      = 4        # minimum block per part on a machine
TARGET_DAYS_INV    = 3        # ideal inventory buffer (days)
MACHINE_STATE_FILE = "machine_state.json"

# Z-score for tentative demand buffer
# Since your tentative is nearly accurate → Z = 1.28 (90% coverage, modest buffer)
# If tentative were very volatile we'd use 1.65 or 2.0
Z_FOR_TENTATIVE    = 1.28

# Priority weights
W_COVERAGE   = 2.0
W_URGENCY    = 3.0
W_RISK       = 1.5
W_D3_BOOST   = 2.0   # how strongly D+3 pre-build need lifts priority

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
daily_path      = "C:/Users/Ex0164/Important codes/Child_for_13march_actual.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"1Smart_APS_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — LOAD DATA
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V4  —  Planning date: {PLANNING_DATE}")
print(f"  Actual demand col   : {ACTUAL_DEMAND_COL}")
print(f"  Tentative demand col: {TENTATIVE_DEMAND_COL}")
print(f"  16th March          : IGNORED today — will plan on 15th")
print(f"{'='*62}\n")

print("Loading data...")
stats        = pd.read_excel(book_path,       sheet_name="Sheet2")
daily        = pd.read_excel(daily_path,      sheet_name="Sheet1")
hz_matrix    = pd.read_excel(matrix_path,     sheet_name="HZ_Matrix")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
hz_co_raw    = pd.read_excel(changeover_path, sheet_name="HZ_Changeover")
vt_co_raw    = pd.read_excel(changeover_path, sheet_name="VT_Changeover")

# =============================================================
# SECTION 5 — VALIDATE DEMAND COLUMNS
# -------------------------------------------------------------
# Columns are read directly by name — no date parsing needed.
# If a column is missing, a clear error is raised showing what
# columns are actually in the file so you know what to fix.
# =============================================================

def validate_column(df, col_name, label):
    """Check the column exists; if not, print all columns and raise."""
    if col_name in df.columns:
        print(f"  {label:25s}: '{col_name}'  ✓")
        return
    raise ValueError(
        f"\n  ERROR: Column not found — {col_name}\n"
        f"  Label : {label}\n"
        f"  Fix   : Update {label.split()[0].upper()}_DEMAND_COL in Section 1\n"
        f"  Columns available in file:\n"
        + "\n".join(f"    '{c}'" for c in df.columns)
    )

print("Validating demand columns...")
validate_column(daily, ACTUAL_DEMAND_COL,    "Actual demand col")
validate_column(daily, TENTATIVE_DEMAND_COL, "Tentative demand col")

# =============================================================
# SECTION 6 — MERGE & PRODUCTION RATE
# =============================================================

data = stats.merge(daily, left_on="Part", right_on="Material")

data["Rate"] = (3600 / data["Cycle time "].replace(0, np.nan)) * data["cavity"]
data = data[data["Rate"].notna()].copy()

# =============================================================
# SECTION 7 — BUILD LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        k: (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
    }

inventory       = safe_dict(data, "Material", "Inventory on 24th")
rate            = safe_dict(data, "Material", "Rate")

# D+1: actual demand — must supply this (15th March)
demand_d1       = safe_dict(data, "Material", ACTUAL_DEMAND_COL)

# D+3: tentative demand — pre-build buffer for this (17th March)
demand_d3_tent  = safe_dict(data, "Material", TENTATIVE_DEMAND_COL)

# Historical stats (for smarter Z-score if available)
mean_demand = (safe_dict(data, "Material", "Mean_Demand")
               if "Mean_Demand" in data.columns else demand_d1.copy())
std_demand  = (safe_dict(data, "Material", "Std_Dev_Demand")
               if "Std_Dev_Demand" in data.columns
               else {k: 0.0 for k in demand_d1})

# =============================================================
# SECTION 7B — CHANGEOVER TIMES PER MACHINE
# -------------------------------------------------------------
# Unique_Machines_vt.xlsx has two sheets:
#   HZ_Changeover: columns "Unique Machines" | "Changeover time"
#   VT_Changeover: columns "Unique Machines" | "Changeover time"
# Changeover time is stored in minutes — converted to hours here.
# =============================================================

def build_changeover_dict(co_df):
    """
    Build {machine_name: changeover_hours} from the changeover sheet.
    Column names: 'Unique Machines' and 'Changeover time' (minutes).
    Falls back to 40 minutes if a machine is missing from the sheet.
    """
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0   # convert to hours
    return co_dict

hz_changeover = build_changeover_dict(hz_co_raw)
vt_changeover = build_changeover_dict(vt_co_raw)

# Fallback changeover in hours if a machine is not in the sheet
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

print(f"\n  HZ changeover times loaded: {len(hz_changeover)} machines")
for m, h in hz_changeover.items():
    print(f"    {m:25s} → {h*60:.0f} min ({h:.3f} h)")
print(f"\n  VT changeover times loaded: {len(vt_changeover)} machines")
for m, h in vt_changeover.items():
    print(f"    {m:25s} → {h*60:.0f} min ({h:.3f} h)")

# =============================================================
# SECTION 7C — PART CATEGORY  (Runner / Repeater / Stranger)
# -------------------------------------------------------------
# Category column lives in both HZ_Matrix and VT_Matrix sheets.
# We merge both and deduplicate — a part should have only one category.
# Runner rule:
#   inv > 1 day  → changeover allowed, plan normally
#   inv ≤ 1 day  → NO changeover, MUST stay on same machine as last run
# =============================================================

def build_category_dict(matrix):
    """Extract {part: category} from a compatibility matrix sheet."""
    cat = {}
    if "Category" in matrix.columns:
        for _, row in matrix.iterrows():
            part = row["Part"]
            val  = str(row["Category"]).strip() if pd.notna(row["Category"]) else "Stranger"
            cat[part] = val
    return cat

hz_category = build_category_dict(hz_matrix)
vt_category = build_category_dict(vt_matrix)
# Merge both — VT takes precedence if a part appears in both
part_category = {**hz_category, **vt_category}

runner_count   = sum(1 for v in part_category.values() if v == "Runner")
repeater_count = sum(1 for v in part_category.values() if v == "Repeater")
stranger_count = sum(1 for v in part_category.values() if v == "Stranger")
print(f"\n  Part categories — Runner: {runner_count}  "
      f"Repeater: {repeater_count}  Stranger: {stranger_count}")

# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# =============================================================

# =============================================================
# SECTION 8 — MACHINE STATE  (last part run per machine)
# -------------------------------------------------------------
# This file is created automatically after the first run.
# On first run (file does not exist yet):
#   → No changeover penalty applied to anyone — clean slate
#   → After planning, the file is written with today's last parts
# From second run onwards:
#   → File is read, changeover penalties apply correctly
#   → Parts that ran last on a machine get zero changeover cost
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        with open(MACHINE_STATE_FILE) as f:
            state = json.load(f)
        print(f"  Machine state loaded  :")
        for m, p in state.items():
            print(f"    {m:25s} last ran → {p}")
        return state
    # First run — no state file exists yet
    print("  Machine state         : FIRST RUN — no previous state")
    print("                          No changeover penalties applied today.")
    print(f"                          State will be saved to '{MACHINE_STATE_FILE}'")
    print("                          after this run for use tomorrow.")
    return {}   # empty dict → machine_last_part[m] = None → no penalty

def save_machine_state(hz_state, vt_state):
    combined = {**hz_state, **vt_state}
    # Remove machines where last part is None (never ran)
    combined = {m: p for m, p in combined.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → {MACHINE_STATE_FILE}")
    print("  (Tomorrow's plan will use this to avoid unnecessary changeovers)")
    for m, p in combined.items():
        print(f"    {m:25s} last ran → {p}")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

hz_compat, hz_machines = build_compatibility(hz_matrix)
vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — DEMAND HORIZON ANALYSIS
# -------------------------------------------------------------
# For each part, calculate:
#
#   d1_shortage   = max(0, D+1 demand − current inventory)
#                 = what we MUST produce today for 15th supply
#
#   d3_buffer     = extra to pre-build today for 17th demand
#                 = max(0, risk_adjusted_D3 − inv_at_d3_start
#                           − machine_capacity_on_16th
#                           − machine_capacity_on_17th)
#
#   today_target  = d1_shortage + d3_buffer
#
# Why do we subtract capacity on 16th AND 17th?
#   Because even without a demand file for 16th, we can still
#   PRODUCE on 16th. So the pre-build needed today is only what
#   BOTH 16th and 17th combined cannot cover.
#
# Note: Since tentative is nearly accurate, Z = 1.28 (modest buffer)
# =============================================================

def compute_demand_horizon(parts, compatibility, machines):

    def max_daily_capacity(part):
        """Max units producible in one full day across all compatible machines."""
        compat_machines = compatibility.get(part, [])
        if not compat_machines:
            return 0
        r = rate.get(part, 1)
        # Each machine can run AVAILABLE_HOURS independently
        return len(compat_machines) * AVAILABLE_HOURS * r

    rows = []
    for p in parts:
        inv      = inventory.get(p, 0)
        d1       = demand_d1.get(p, 0)
        d3_t     = demand_d3_tent.get(p, 0)
        mean     = mean_demand.get(p, max(d1, 1))
        std      = std_demand.get(p, 0)
        r        = rate.get(p, 1)

        # ── D+1 (15th) shortage ─────────────────────────────
        d1_shortage    = max(0.0, d1 - inv)
        inv_after_d1   = max(0.0, inv - d1)   # inventory left after supplying 15th

        # ── D+3 (17th) risk-adjusted target ─────────────────
        # Since tentative is nearly accurate, small Z = 1.28
        d3_risk_adj    = d3_t + Z_FOR_TENTATIVE * std

        # ── Capacity available on 16th and 17th ─────────────
        # 16th: no demand file, machines run freely → full capacity available
        # 17th: machines run freely → full capacity available
        # Together they can cover: 2 × daily_capacity
        cap_16th       = max_daily_capacity(p)
        cap_17th       = max_daily_capacity(p)

        # Inventory at start of 17th =
        #   inv_after_d1 (left after 15th supply)
        #   + whatever we produce on 16th (up to cap_16th, but only if needed)
        # We assume 16th production goes toward 17th demand as well
        # So total coverage for 17th = inv_after_d1 + cap_16th + cap_17th

        total_coverage_for_d3 = inv_after_d1 + cap_16th + cap_17th

        # Pre-build needed today = gap that even 16th+17th machines can't cover
        d3_buffer = max(0.0, d3_risk_adj - total_coverage_for_d3)

        # ── Combined today's target ──────────────────────────
        today_target = d1_shortage + d3_buffer

        # ── D+3 status flag ─────────────────────────────────
        if d3_buffer > 0:
            d3_flag = "PRE-BUILD NEEDED"
        elif d3_risk_adj > (cap_16th + cap_17th) * 0.8:
            d3_flag = "WATCH — near capacity"
        else:
            d3_flag = "OK"

        rows.append({
            "Part":                   p,
            "Inventory_Now":          round(inv, 0),

            # D+1 (15th) columns
            "D1_Actual_15th":         round(d1, 0),
            "D1_Shortage":            round(d1_shortage, 0),
            "Inv_After_15th_Supply":  round(inv_after_d1, 0),

            # D+3 (17th) columns
            "D3_Tentative_17th":      round(d3_t, 0),
            "D3_Risk_Adjusted":       round(d3_risk_adj, 0),
            "Cap_16th_Available":     round(cap_16th, 0),
            "Cap_17th_Available":     round(cap_17th, 0),
            "D3_Pre_Build_Buffer":    round(d3_buffer, 0),
            "D3_Flag":                d3_flag,

            # Combined
            "Today_Total_Target_Qty": round(today_target, 0),
            "Today_Total_Target_Hrs": round(today_target / r if r > 0 else 0, 2),
        })

    return pd.DataFrame(rows)

# =============================================================
# SECTION 11 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv  = inventory.get(p, 0)
        d1   = demand_d1.get(p, 0)
        mean = mean_demand.get(p, d1)
        # Parts with zero demand are not critical regardless of inventory
        if d1 == 0 and mean == 0:
            continue
        daily = max(mean, d1, 1)
        coverage.append(inv / daily)

    if not coverage:
        # All parts have zero demand — best case
        return 3, 1.28, "SCENARIO 3 — All parts healthy (zero demand today)"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, 2.00, "SCENARIO 0 — ALL parts critical (inv < 1 day demand)"
    elif critical > 0:
        return 1, 1.65, f"SCENARIO 1 — {critical}/{n} parts critical (inv < 1 day)"
    elif low > 0:
        return 2, 1.28, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, 1.28, f"SCENARIO 3 — All parts healthy (≥{TARGET_DAYS_INV} days inv)"

# =============================================================
# SECTION 12 — PRIORITY SCORING  (D+3-aware)
# =============================================================

def compute_priority(parts, horizon_df):
    horizon = horizon_df.set_index("Part")
    rows    = []

    for p in parts:
        inv  = inventory.get(p, 0)
        d1   = demand_d1.get(p, 0)
        mean = mean_demand.get(p, max(d1, 1))
        std  = std_demand.get(p, 0)

        days_cov = inv / mean if mean > 0 else 999

        # Urgency based on D+1 coverage
        if days_cov < 1:
            urgency = 1.0
        elif days_cov < 2:
            urgency = 0.5
        else:
            urgency = 0.0

        cv = std / mean if mean > 0 else 0

        # D+3 boost: if pre-build buffer > 0, add extra urgency
        d3_buf  = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
        d3_boost = min(1.0, d3_buf / max(mean, 1))

        score = (W_COVERAGE * (1.0 / (days_cov + 0.01))
               + W_URGENCY  * urgency
               + W_RISK     * cv
               + W_D3_BOOST * d3_boost)

        today_target = (horizon.loc[p, "Today_Total_Target_Qty"]
                        if p in horizon.index else d1)

        rows.append({
            "Part":           p,
            "Days_Coverage":  round(days_cov, 2),
            "Urgency_D1":     urgency,
            "CV":             round(cv, 3),
            "D3_Boost":       round(d3_boost, 3),
            "Score":          round(score, 4),
            "D1_Demand_15th": round(d1, 0),
            "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            "D3_Pre_Build":   round(d3_buf, 0),
            "Today_Target":   round(today_target, 0),
        })

    return (pd.DataFrame(rows)
              .sort_values("Score", ascending=False)
              .reset_index(drop=True))

# =============================================================
# SECTION 13 — UBER-STYLE MACHINE RANKER  (category-aware)
# =============================================================

def rank_machines(part, machines, compatibility, machine_hours,
                  machine_last_part, changeover_dict, inv_days):
    """
    Returns machines sorted by assignment cost (lowest = best choice).

    Changeover logic:
      - Changeover time is now machine-specific (from Unique_Machines_vt.xlsx)
      - If same part ran last on machine → 0 changeover hours (no penalty)
      - If different part ran last       → machine's own changeover time

    Runner rule:
      - inv_days > 1  → standard logic, changeover allowed
      - inv_days ≤ 1  → NO changeover allowed
                        ONLY the machine where this part last ran is eligible
                        If that machine has no free hours → empty list returned
                        → part goes to Not_Planned with specific Runner reason
    """
    category    = part_category.get(part, "Stranger")
    is_runner   = (category == "Runner")
    runner_lock = is_runner and inv_days <= 1.0   # enforce same-machine rule

    ranked = []
    for m in machines:
        if m not in compatibility.get(part, []):
            continue

        used = machine_hours.get(m, 0)
        free = AVAILABLE_HOURS - used
        if free < MIN_RUN_HOURS:
            continue

        last = machine_last_part.get(m)

        # Runner with low inventory → only its last machine is allowed
        if runner_lock:
            if last != part:
                continue   # skip every machine where this part didn't last run

        # Changeover cost in hours for this specific machine
        if last == part:
            co_hrs = 0.0                                          # no changeover
        else:
            co_hrs = changeover_dict.get(m, DEFAULT_CHANGEOVER_HRS)

        # Effective free hours = total free − changeover cost
        effective_free = free - co_hrs
        if effective_free < MIN_RUN_HOURS:
            continue   # after changeover, not enough time for minimum run

        cost = co_hrs / AVAILABLE_HOURS + (used / AVAILABLE_HOURS)
        ranked.append((m, cost, effective_free, co_hrs))

    ranked.sort(key=lambda x: x[1])
    return ranked, runner_lock

# =============================================================
# SECTION 14 — 22-HOUR UTILIZATION FILLER
# =============================================================

def fill_remaining_hours(plan, machine_hours, machine_last_part,
                         parts, compatibility, machines,
                         current_inventory, horizon_df, scenario):
    """
    After primary assignments, fill every remaining minute of machine time.

    The MIN_RUN_HOURS (4h) check applies only to PRIMARY scheduling
    (first-time assignment of a new part to a machine).
    The filler has NO minimum — it uses whatever hours remain, even 1h or 2h,
    by extending an already-running part (zero changeover, any duration is fine).

    This fixes the 18-22h gap: a machine at 18.5h with 3.5h left would
    previously be skipped (3.5 < 4h minimum). Now it gets extended.
    """
    filler  = []
    horizon = horizon_df.set_index("Part")

    for m in machines:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
        if remaining <= 0:
            continue   # truly full — nothing to add

        on_machine = [row["Part"] for row in plan if row["Machine"] == m]

        # ── Option A: extend already-planned part ────────────
        # No minimum time check here — any remaining hours are useful.
        # Prefer the part with the largest gap to its 3-day inventory target.
        extended = False
        best_part, best_gap = None, -1
        for p in on_machine:
            inv_now    = current_inventory.get(p, 0)
            target_inv = mean_demand.get(p, demand_d1.get(p, 0)) * TARGET_DAYS_INV
            d3_buf     = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
            gap        = max(0, target_inv - inv_now) + d3_buf
            if gap > best_gap:
                best_gap, best_part = gap, p

        if best_part and best_gap > 0:
            p         = best_part
            r_val     = rate.get(p, 1)
            extra_qty = round(remaining * r_val, 0)
            machine_hours[m]    += remaining
            current_inventory[p] = current_inventory.get(p, 0) + extra_qty
            filler.append({
                "Part":           p,
                "Machine":        m,
                "Run_Hours":      round(remaining, 2),
                "Production_Qty": extra_qty,
                "Type":           ("Extend — D3 buffer"
                                   if (p in horizon.index and horizon.loc[p, "D3_Pre_Build_Buffer"] > 0)
                                   else "Extend — inv build"),
                "Changeover":     "No",
                "D1_Demand_15th": round(demand_d1.get(p, 0), 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            })
            extended = True

        if extended:
            continue

        # ── Option B: new part — only if remaining >= MIN_RUN_HOURS
        # For a BRAND NEW part on this machine we still require 4h minimum
        # because a changeover + very short run is wasteful.
        if remaining < MIN_RUN_HOURS:
            continue   # not enough time for a new part — leave as-is

        candidates = []
        for p in parts:
            if p in on_machine:
                continue
            if m not in compatibility.get(p, []):
                continue
            inv_now = current_inventory.get(p, 0)
            d3_buf  = horizon.loc[p, "D3_Pre_Build_Buffer"] if p in horizon.index else 0
            tgt_inv = mean_demand.get(p, demand_d1.get(p, 0)) * TARGET_DAYS_INV
            if scenario == 3 and inv_now >= tgt_inv and d3_buf == 0:
                continue
            candidates.append((p, d3_buf))

        candidates.sort(key=lambda x: -x[1])

        for p, d3_buf in candidates:
            last            = machine_last_part.get(m)
            changeover_flag = "No" if last == p else "Yes"
            r_val           = rate.get(p, 1)
            run_h           = remaining
            qty             = round(run_h * r_val, 0)

            machine_hours[m]    += run_h
            current_inventory[p] = current_inventory.get(p, 0) + qty
            machine_last_part[m] = p

            filler.append({
                "Part":           p,
                "Machine":        m,
                "Run_Hours":      round(run_h, 2),
                "Production_Qty": qty,
                "Type":           "New — D3 pre-build" if d3_buf > 0 else "New — inv fill",
                "Changeover":     changeover_flag,
                "D1_Demand_15th": round(demand_d1.get(p, 0), 0),
                "D3_Tent_17th":   round(demand_d3_tent.get(p, 0), 0),
            })
            break

    return filler

# =============================================================
# SECTION 15 — MAIN SCHEDULER
# =============================================================

def schedule(parts, compatibility, machines, changeover_dict, label=""):

    print(f"\n{'─'*62}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(machines)} machines")
    print(f"{'─'*62}")

    # Scenario
    scenario, _, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    # Demand horizon
    horizon_df = compute_demand_horizon(parts, compatibility, machines)

    pre_build_parts = horizon_df[horizon_df["D3_Pre_Build_Buffer"] > 0]
    if not pre_build_parts.empty:
        print(f"\n  D+3 pre-build required ({len(pre_build_parts)} parts):")
        for _, r in pre_build_parts.iterrows():
            print(f"    {r['Part']:30s}  "
                  f"15th={r['D1_Actual_15th']:.0f}  "
                  f"17th_tent={r['D3_Tentative_17th']:.0f}  "
                  f"17th_risk={r['D3_Risk_Adjusted']:.0f}  "
                  f"pre_build={r['D3_Pre_Build_Buffer']:.0f}")
    else:
        print("  No D+3 pre-build needed — 16th+17th capacity covers 17th demand")

    # State
    machine_hours     = {m: 0.0 for m in machines}
    machine_last_part = {m: machine_state.get(m) for m in machines}
    current_inventory = inventory.copy()

    plan, deferred, not_planned = [], [], []

    # Priority sort
    priority_df = compute_priority(parts, horizon_df)
    horizon_idx = horizon_df.set_index("Part")

    print(f"\n  Priority order:")
    for _, r in priority_df.iterrows():
        print(f"    {r['Part']:30s}  "
              f"score={r['Score']:.3f}  "
              f"cov={r['Days_Coverage']}d  "
              f"D1_short={max(0, r['D1_Demand_15th'] - inventory.get(r['Part'], 0)):.0f}  "
              f"D3_prebuild={r['D3_Pre_Build']:.0f}")

    # ── Main assignment loop ────────────────────────────────
    print(f"\n  Assignments:")
    for _, row in priority_df.iterrows():
        part     = row["Part"]
        inv_now  = current_inventory.get(part, 0)
        d1       = demand_d1.get(part, 0)
        d3_buf   = row["D3_Pre_Build"]
        r_val    = rate.get(part, 1)

        inv_covers_d1  = inv_now >= d1
        compatible_mch = compatibility.get(part, [])
        days_cov_now   = inv_now / max(mean_demand.get(part, max(d1, 1)), 1)

        # ── Decision 1: zero demand — never needs planning ─────
        # Regardless of scenario, if demand is 0 and no D+3 buffer
        # is needed, this part should not be planned today.
        if d1 == 0 and d3_buf == 0:
            deferred.append({
                "Part":              part,
                "Inventory_Now":     round(inv_now, 0),
                "D1_Demand_15th":    0,
                "Inventory_Surplus": round(inv_now, 0),
                "Days_Coverage":     round(days_cov_now, 2),
                "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                "Reason":            "Demand is zero — no production required today",
                "Next_Action":       "Will be re-evaluated when demand appears",
            })
            print(f"    - {part:30s}  NOT REQUIRED — demand = 0")
            continue

        # ── Decision 2: inventory sufficient, scenario not critical
        # In scenario 0/1 (critical), we MUST try to produce everything
        # even if inventory partially covers demand — machines must run.
        # In scenario 2/3 (healthy/moderate), if inventory covers D+1
        # and no D+3 buffer needed → safe to skip today.
        if inv_covers_d1 and d3_buf == 0 and scenario >= 2:
            deferred.append({
                "Part":              part,
                "Inventory_Now":     round(inv_now, 0),
                "D1_Demand_15th":    round(d1, 0),
                "Inventory_Surplus": round(inv_now - d1, 0),
                "Days_Coverage":     round(days_cov_now, 2),
                "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                "Reason":            "Inventory sufficient — covers D+1 demand with surplus",
                "Next_Action":       "Will be re-evaluated tomorrow",
            })
            print(f"    - {part:30s}  NOT REQUIRED TODAY"
                  f"  inv={inv_now:.0f}  d1={d1:.0f}  surplus={inv_now-d1:.0f}")
            continue

        # ── Decision 3: scenario 0/1 but inventory already covers D+1
        # AND no D+3 buffer needed — still skip, log separately
        if inv_covers_d1 and d3_buf == 0 and scenario < 2:
            deferred.append({
                "Part":              part,
                "Inventory_Now":     round(inv_now, 0),
                "D1_Demand_15th":    round(d1, 0),
                "Inventory_Surplus": round(inv_now - d1, 0),
                "Days_Coverage":     round(days_cov_now, 2),
                "D3_Tent_17th":      round(demand_d3_tent.get(part, 0), 0),
                "Reason":            "Inventory sufficient even in critical scenario — skipped to free capacity for critical parts",
                "Next_Action":       "Monitor — will be produced once critical parts are covered",
            })
            print(f"    - {part:30s}  NOT REQUIRED TODAY (critical scenario but inv covers D+1)")
            continue

        # ── Decision 4: part needs production — find a machine ─
        category  = part_category.get(part, "Stranger")
        inv_days  = inv_now / max(mean_demand.get(part, max(d1, 1)), 1)

        today_qty = row["Today_Target"]
        if scenario == 0:
            today_qty = max(0, d1 - inv_now)
        target_hrs = max(MIN_RUN_HOURS, today_qty / r_val if r_val > 0 else MIN_RUN_HOURS)

        # Check if part even has compatible machines defined
        if not compatible_mch:
            not_planned.append({
                "Part":                part,
                "Category":            category,
                "D1_Demand_15th":      round(d1, 0),
                "Inventory_Now":       round(inv_now, 0),
                "Inv_Days_Coverage":   round(inv_days, 2),
                "Shortage":            round(max(0, d1 - inv_now), 0),
                "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                "Compatible_Machines": "NONE DEFINED",
                "Reason":              "Part has no compatible machines in the matrix",
                "Action_Needed":       "Add this part to the compatibility matrix",
            })
            print(f"    ✗ {part:30s}  NOT PLANNED — not in compatibility matrix")
            continue

        # Uber-style machine ranking — category + per-machine changeover aware
        ranked, runner_lock = rank_machines(
            part, machines, compatibility,
            machine_hours, machine_last_part,
            changeover_dict, inv_days
        )

        # No eligible machine found — determine precise reason
        if not ranked:
            machines_status  = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                                for m in compatible_mch if m in machine_hours}
            not_in_group     = [m for m in compatible_mch if m not in machine_hours]

            if runner_lock:
                last_machine = next(
                    (m for m in compatible_mch if machine_last_part.get(m) == part), None)
                if last_machine:
                    free_on_last = round(AVAILABLE_HOURS - machine_hours.get(last_machine, 0), 2)
                    co_needed    = changeover_dict.get(last_machine, DEFAULT_CHANGEOVER_HRS)
                    reason  = (f"RUNNER (inv={round(inv_days,2)}d < 1 day) — must stay on "
                               f"{last_machine} but only {free_on_last}h free "
                               f"(need {MIN_RUN_HOURS + co_needed:.2f}h after {co_needed*60:.0f}min changeover)")
                    action  = (f"Free capacity on {last_machine} by reducing lower-priority parts, "
                               f"or verify inventory can cover today's demand gap")
                else:
                    reason  = (f"RUNNER (inv={round(inv_days,2)}d < 1 day) — "
                               f"no machine state found for first run, cannot enforce same-machine rule")
                    action  = "Machine state will be saved after this run — rule enforces from tomorrow"
            elif not_in_group:
                reason  = f"Compatible machines {not_in_group} not in this machine group"
                action  = "Check compatibility matrix — machine may belong to wrong group (HZ/VT)"
            elif all(h < MIN_RUN_HOURS for h in machines_status.values()):
                reason  = (f"All compatible machines fully utilised. Free hours: "
                           + ", ".join(f"{m}={h}h" for m, h in machines_status.items()))
                action  = "Reduce lower-priority part qtys or plan this part tomorrow"
            else:
                reason  = (f"Remaining hours exist but < MIN_RUN_HOURS after changeover deduction. "
                           f"Free: " + ", ".join(f"{m}={h}h" for m, h in machines_status.items()))
                action  = "Inventory must cover gap — check Inventory_Health sheet"

            not_planned.append({
                "Part":                part,
                "Category":            category,
                "D1_Demand_15th":      round(d1, 0),
                "Inventory_Now":       round(inv_now, 0),
                "Inv_Days_Coverage":   round(inv_days, 2),
                "Shortage":            round(max(0, d1 - inv_now), 0),
                "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                "Compatible_Machines": ", ".join(compatible_mch),
                "Reason":              reason,
                "Action_Needed":       action,
            })
            print(f"    ✗ {part:30s}  [{category}]  NOT PLANNED — {reason[:70]}")
            continue

        assigned = False

        for m, cost, effective_free, co_hrs in ranked:
            run_h = min(target_hrs, effective_free)
            run_h = max(run_h, MIN_RUN_HOURS)
            run_h = min(run_h, effective_free)
            qty   = round(run_h * r_val, 0)

            machine_hours[m]        += (co_hrs + run_h)
            current_inventory[part]  = current_inventory.get(part, 0) + qty
            machine_last_part[m]     = part

            plan.append({
                "Part":            part,
                "Category":        category,
                "Machine":         m,
                "Run_Hours":       round(run_h, 2),
                "Changeover_Hrs":  round(co_hrs, 3),
                "Total_Hrs_Used":  round(co_hrs + run_h, 2),
                "Production_Qty":  qty,
                "D1_Demand_15th":  round(d1, 0),
                "D3_Tent_17th":    round(demand_d3_tent.get(part, 0), 0),
                "D3_Pre_Build":    round(d3_buf, 0),
                "Changeover":      "No" if co_hrs == 0 else "Yes",
                "Type":            "Primary",
                "Runner_Lock":     "YES" if runner_lock else "No",
                "Priority_Score":  row["Score"],
            })

            co_str = "No" if co_hrs == 0 else f"Yes ({co_hrs*60:.0f}min)"
            print(f"    ✓ {part:30s} [{category:8s}] → {m:15s}  "
                  f"{run_h:.2f}h  qty={qty:>8.0f}  CO={co_str}")
            assigned = True
            break

        if not assigned:
            remaining_target = target_hrs
            split_done       = False

            for m, cost, effective_free, co_hrs in ranked:
                if effective_free < MIN_RUN_HOURS or remaining_target <= 0:
                    break
                run_h = min(remaining_target, effective_free)
                run_h = max(run_h, MIN_RUN_HOURS)
                qty   = round(run_h * r_val, 0)

                machine_hours[m]        += (co_hrs + run_h)
                current_inventory[part]  = current_inventory.get(part, 0) + qty
                machine_last_part[m]     = part
                remaining_target        -= run_h

                plan.append({
                    "Part":           part,
                    "Category":       category,
                    "Machine":        m,
                    "Run_Hours":      round(run_h, 2),
                    "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_h, 2),
                    "Production_Qty": qty,
                    "D1_Demand_15th": round(d1, 0),
                    "D3_Tent_17th":   round(demand_d3_tent.get(part, 0), 0),
                    "D3_Pre_Build":   round(d3_buf, 0),
                    "Changeover":     "No" if co_hrs == 0 else "Yes",
                    "Type":           "Split",
                    "Runner_Lock":    "YES" if runner_lock else "No",
                    "Priority_Score": row["Score"],
                })
                print(f"    ↔ {part:30s} [{category:8s}] → {m:15s}  {run_h:.2f}h  [SPLIT]")
                split_done = True

            if not split_done:
                machines_status = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                                   for m in compatible_mch if m in machine_hours}
                reason = ("Capacity too fragmented after changeover deduction. Free hours: "
                          + ", ".join(f"{m}={h}h" for m, h in machines_status.items()))
                not_planned.append({
                    "Part":                part,
                    "Category":            category,
                    "D1_Demand_15th":      round(d1, 0),
                    "Inventory_Now":       round(inv_now, 0),
                    "Inv_Days_Coverage":   round(inv_days, 2),
                    "Shortage":            round(max(0, d1 - inv_now), 0),
                    "D3_Tent_17th":        round(demand_d3_tent.get(part, 0), 0),
                    "Compatible_Machines": ", ".join(compatible_mch),
                    "Reason":              reason,
                    "Action_Needed":       "Check Inventory_Health sheet",
                })
                print(f"    ✗ {part:30s}  [{category}]  NOT PLANNED — fragmented capacity")

    # 22-hour filler
    print(f"\n  22-hour filler:")
    filler_rows = fill_remaining_hours(
        plan, machine_hours, machine_last_part,
        list(parts), compatibility, machines,
        current_inventory, horizon_df, scenario
    )
    if filler_rows:
        for fr in filler_rows:
            plan.append(fr)
            print(f"    + {fr['Part']:30s} → {fr['Machine']:15s}  "
                  f"{fr['Run_Hours']}h  [{fr['Type']}]  CO={fr['Changeover']}")
    else:
        print("    All machines fully utilised — no filler needed")

    # ── Inventory health after today's plan ─────────────────
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        d1       = demand_d1.get(p, 0)
        d3_t     = demand_d3_tent.get(p, 0)
        produced = sum(r["Production_Qty"] for r in plan if r["Part"] == p)

        # final_inventory = inventory + produced − demand_supplied
        inv_after = inv_b + produced - d1
        mean      = mean_demand.get(p, max(d1, 1))
        days_cov  = inv_after / mean if mean > 0 else 0

        # How much of 17th tentative is covered by this inventory?
        d3_covered  = min(inv_after, d3_t)
        d3_still_gap = max(0, d3_t - inv_after)

        inv_rows.append({
            "Part":                   p,
            "Inv_Before":             round(inv_b, 0),
            "Produced_14th":          round(produced, 0),
            "D1_Supplied_15th":       round(d1, 0),
            "Inv_After_15th_Supply":  round(inv_after, 0),   # = inv + produced - d1
            "Days_Coverage":          round(days_cov, 2),
            "D3_Tentative_17th":      round(d3_t, 0),
            "D3_Covered_By_Inv":      round(d3_covered, 0),
            "D3_Still_Needs_16th_17th": round(d3_still_gap, 0),
            "Status":                 ("OK"       if days_cov >= 1
                                       else "CRITICAL" if inv_after < 0
                                       else "LOW"),
        })

    # Machine utilization — rich columns including parts list
    mach_rows = []
    for m in machines:
        used       = machine_hours.get(m, 0)
        remaining  = AVAILABLE_HOURS - used
        parts_run  = [r["Part"] for r in plan if r["Machine"] == m]
        parts_str  = ", ".join(parts_run) if parts_run else "— idle —"
        co_count   = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")

        if used >= AVAILABLE_HOURS - 0.5:
            util_status = "FULL"
        elif used >= AVAILABLE_HOURS * 0.85:
            util_status = "GOOD"
        elif used >= AVAILABLE_HOURS * 0.5:
            util_status = "PARTIAL"
        else:
            util_status = "UNDERUSED"

        mach_rows.append({
            "Machine":           m,
            "Total_Available_Hrs": AVAILABLE_HOURS,
            "Used_Hours":        round(used, 2),
            "Unused_Hours":      round(remaining, 2),
            "Utilization_%":     round(used / AVAILABLE_HOURS * 100, 1),
            "Status":            util_status,
            "Parts_Planned":     len(parts_run),
            "Changeovers":       co_count,
            "Last_Part_Run":     machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": parts_str,
        })

    plan_df     = pd.DataFrame(plan)        if plan        else pd.DataFrame()
    def_df      = pd.DataFrame(deferred)    if deferred    else pd.DataFrame()
    not_df      = pd.DataFrame(not_planned) if not_planned else pd.DataFrame()
    mach_df     = pd.DataFrame(mach_rows)
    inv_df      = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)

    return plan_df, def_df, not_df, mach_df, inv_df, machine_last_part, horizon_df

# =============================================================
# SECTION 16 — RUN BOTH MACHINE GROUPS
# =============================================================

hz_parts = data[data["Material"].isin(hz_matrix["Part"])]["Material"].unique()
vt_parts = data[data["Material"].isin(vt_matrix["Part"])]["Material"].unique()

hz_plan, hz_def, hz_not, hz_mach, hz_inv, hz_state, hz_horizon = \
    schedule(hz_parts, hz_compat, hz_machines, hz_changeover, "HZ Machines")

vt_plan, vt_def, vt_not, vt_mach, vt_inv, vt_state, vt_horizon = \
    schedule(vt_parts, vt_compat, vt_machines, vt_changeover, "VT Machines")

save_machine_state(hz_state, vt_state)

# =============================================================
# SECTION 17 — SAVE OUTPUT
# =============================================================

# =============================================================
# SECTION 17 — SAVE OUTPUT  (formatted Excel)
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Colour palette for header rows
HEADER_COLORS = {
    "HZ_Plan":               "1F4E79",   # dark blue
    "VT_Plan":               "1F4E79",
    "HZ_Machine_Util":       "375623",   # dark green
    "VT_Machine_Util":       "375623",
    "HZ_Not_Planned":        "7B2C2C",   # dark red
    "VT_Not_Planned":        "7B2C2C",
    "HZ_Not_Required_Today":           "7F6000",   # dark amber
    "VT_Not_Required_Today":           "7F6000",
    "HZ_Inventory_Health":   "4A235A",   # dark purple
    "VT_Inventory_Health":   "4A235A",
    "HZ_Demand_Horizon":     "154360",
    "VT_Demand_Horizon":     "154360",
    "ALL_Plan_Combined":     "1F4E79",
}

STATUS_FILLS = {
    "FULL":      PatternFill("solid", fgColor="C6EFCE"),   # green
    "GOOD":      PatternFill("solid", fgColor="DDEBF7"),   # blue
    "PARTIAL":   PatternFill("solid", fgColor="FFEB9C"),   # yellow
    "UNDERUSED": PatternFill("solid", fgColor="FFC7CE"),   # red
    "OK":        PatternFill("solid", fgColor="C6EFCE"),
    "LOW":       PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":  PatternFill("solid", fgColor="FFC7CE"),
    "PRE-BUILD NEEDED": PatternFill("solid", fgColor="FFC7CE"),
    "WATCH — near capacity": PatternFill("solid", fgColor="FFEB9C"),
}

def style_sheet(ws, header_hex):
    """Apply header formatting and auto column widths to a worksheet."""
    header_fill = PatternFill("solid", fgColor=header_hex)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    center      = Alignment(horizontal="center", vertical="center", wrap_text=True)

    # Style header row
    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center

    ws.row_dimensions[1].height = 32

    # Auto-fit column widths based on content
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                cell_len = len(str(cell.value)) if cell.value is not None else 0
                max_len  = max(max_len, cell_len)
            except Exception:
                pass
        # Cap between 10 and 50 characters wide
        ws.column_dimensions[col_letter].width = max(10, min(50, max_len + 3))

    # Colour status/flag cells
    status_col_names = {"Status", "D3_Flag", "Utilization_Status"}
    header_row = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(header_row, start=1):
        if col_name in status_col_names or "Status" in str(col_name):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value), None)
                    if fill:
                        cell.fill = fill

    # Freeze the header row
    ws.freeze_panes = "A2"

print(f"\nWriting → {output_path}")

all_plan = pd.concat([hz_plan, vt_plan], ignore_index=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # ── Plans ────────────────────────────────────────────────
    hz_plan.to_excel(writer, sheet_name="HZ_Plan",           index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan",           index=False)

    # ── Machine utilization ──────────────────────────────────
    hz_mach.to_excel(writer, sheet_name="HZ_Machine_Util",   index=False)
    vt_mach.to_excel(writer, sheet_name="VT_Machine_Util",   index=False)

    # ── Not planned (capacity exhausted) ────────────────────
    hz_not.to_excel(writer,  sheet_name="HZ_Not_Planned",    index=False)
    vt_not.to_excel(writer,  sheet_name="VT_Not_Planned",    index=False)

    # ── Deferred (inventory sufficient) ─────────────────────
    hz_def.to_excel(writer,  sheet_name="HZ_Not_Required_Today",       index=False)
    vt_def.to_excel(writer,  sheet_name="VT_Not_Required_Today",       index=False)

    # ── Inventory health ─────────────────────────────────────
    hz_inv.to_excel(writer,  sheet_name="HZ_Inventory_Health", index=False)
    vt_inv.to_excel(writer,  sheet_name="VT_Inventory_Health", index=False)

    # ── Demand horizon ───────────────────────────────────────
    hz_horizon.to_excel(writer, sheet_name="HZ_Demand_Horizon", index=False)
    vt_horizon.to_excel(writer, sheet_name="VT_Demand_Horizon", index=False)

    # ── Combined plan ────────────────────────────────────────
    all_plan.to_excel(writer, sheet_name="ALL_Plan_Combined", index=False)

# Apply formatting after writing (openpyxl post-process)
wb = load_workbook(output_path)
for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames:
        style_sheet(wb[sheet_name], header_hex)

# Set sheet tab order — most important sheets first
tab_order = [
    "HZ_Plan", "VT_Plan",
    "HZ_Machine_Util", "VT_Machine_Util",
    "HZ_Not_Planned", "VT_Not_Planned",
    "HZ_Not_Required_Today", "VT_Not_Required_Today",
    "HZ_Inventory_Health", "VT_Inventory_Health",
    "HZ_Demand_Horizon", "VT_Demand_Horizon",
    "ALL_Plan_Combined",
]
for i, name in enumerate(tab_order):
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = (
            "1F4E79" if "Plan" in name else
            "375623" if "Machine" in name else
            "7B2C2C" if "Not_Planned" in name else
            "7F6000" if "Deferred" in name else
            "4A235A"
        )

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 18 — SUMMARY PRINT
# =============================================================

print(f"\n{'='*62}")
print(f"  Smart APS V4 Complete  —  {PLANNING_DATE}")
print(f"{'='*62}")
print(f"  HZ  planned={len(hz_plan):>4}  not_required_today={len(hz_def):>4}  not_planned={len(hz_not):>4}")
print(f"  VT  planned={len(vt_plan):>4}  not_required_today={len(vt_def):>4}  not_planned={len(vt_not):>4}")
print(f"\n  not_required_today = inventory already sufficient, no action needed")
print(f"  not_planned        = production was needed but no machine capacity available")

all_horizon = pd.concat([hz_horizon, vt_horizon], ignore_index=True)
pre_build   = all_horizon[all_horizon["D3_Pre_Build_Buffer"] > 0]

if not pre_build.empty:
    print(f"\n  D+3 (17th) pre-build buffers applied to {len(pre_build)} parts:")
    print(f"  {'Part':<30}  {'15th demand':>12}  {'17th tent':>10}  {'17th risk':>10}  {'buffer':>8}")
    print(f"  {'─'*30}  {'─'*12}  {'─'*10}  {'─'*10}  {'─'*8}")
    for _, r in pre_build.iterrows():
        print(f"  {r['Part']:<30}  {r['D1_Actual_15th']:>12.0f}  "
              f"{r['D3_Tentative_17th']:>10.0f}  "
              f"{r['D3_Risk_Adjusted']:>10.0f}  "
              f"{r['D3_Pre_Build_Buffer']:>8.0f}")
else:
    print(f"\n  No D+3 pre-build needed — 16th and 17th machine capacity is sufficient")

print(f"\n  Output  → {output_path}")
print(f"  State   → {MACHINE_STATE_FILE}")
print(f"\n  NOTE: Change the 3 lines in SECTION 1 each morning before running.")
print(f"        Tomorrow (15th): ACTUAL_DEMAND_COL    = '2026-03-13 Total Production Plan'")
print(f"                         TENTATIVE_DEMAND_COL = '2026-03-15 Total Production Plan'")
print(f"                         PLANNING_DATE        = date(2026, 3, 15)")


  Smart APS V4  —  Planning date: 2026-03-14
  Actual demand col   : 2026-03-12 Total Production Plan
  Tentative demand col: 2026-03-14 Total Production Plan
  16th March          : IGNORED today — will plan on 15th

Loading data...
Validating demand columns...
  Actual demand col        : '2026-03-12 Total Production Plan'  ✓
  Tentative demand col     : '2026-03-14 Total Production Plan'  ✓

  HZ changeover times loaded: 29 machines
    BOY-10T-I M027            → 20 min (0.333 h)
    BOY-10T-II M028           → 20 min (0.333 h)
    BOY-10T-III M029          → 20 min (0.333 h)
    BOY-22T-I M030            → 30 min (0.500 h)
    BOY-22T-II M031           → 30 min (0.500 h)
    BOY-22T-III M032          → 30 min (0.500 h)
    BOY-22T-IV M033           → 30 min (0.500 h)
    BOY-22T-V M050            → 30 min (0.500 h)
    BOY-22T-VI M051           → 30 min (0.500 h)
    FANUC-50T-I M035          → 30 min (0.500 h)
    FANUC-50T-II M036         → 30 min (0.500 h)
    FANUC-50T-III M0